### Step 120 — Dataset Loading and Structural Validation

**Objective:** Verify dataset availability, structure, and consistency of request identifiers.


In [8]:
# Step 120 - Load and inspect project datasets

import pandas as pd
from pathlib import Path

# Define project paths
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"

product_path = DATA_DIR / "processed" / "product_master.csv"
ground_truth_path = DATA_DIR / "processed" / "request_ground_truth_3000.csv"
customer_path = DATA_DIR / "interim" / "llm_customer_requests_3000.csv"

# Check file existence
for path in [product_path, ground_truth_path, customer_path]:
    print(f"{path.name}: {'FOUND' if path.is_file() else 'MISSING'}")

# Load datasets
product_master = pd.read_csv(product_path)
ground_truth = pd.read_csv(ground_truth_path)
customer_requests = pd.read_csv(customer_path)

print("\n=== DATASET OVERVIEW ===")

print("Product Master:", product_master.shape)
print("Ground Truth:", ground_truth.shape)
print("Customer Requests:", customer_requests.shape)

print("\n=== COLUMN NAMES ===")

print("\nProduct Master:")
print(product_master.columns.tolist())

print("\nGround Truth:")
print(ground_truth.columns.tolist())

print("\nCustomer Requests:")
print(customer_requests.columns.tolist())

print("\n=== REQUEST ID VALIDATION ===")

gt_ids = set(ground_truth["REQUEST_ID"].astype(str))
customer_ids = set(customer_requests["REQUEST_ID"].astype(str))

print("Unique Ground Truth IDs:", len(gt_ids))
print("Unique Customer IDs:", len(customer_ids))
print("Missing customer texts:", len(gt_ids - customer_ids))
print("Customers without Ground Truth:", len(customer_ids - gt_ids))

assert gt_ids == customer_ids, "REQUEST_ID mismatch detected!"

print("\nSTEP 120: PASSED")

product_master.csv: FOUND
request_ground_truth_3000.csv: FOUND
llm_customer_requests_3000.csv: FOUND

=== DATASET OVERVIEW ===
Product Master: (500, 25)
Ground Truth: (12021, 6)
Customer Requests: (3000, 4)

=== COLUMN NAMES ===

Product Master:
['PRODUCT_ID', 'WIDTH_CM', 'OVEN_FUNCTIONS', 'OVEN_VOLUME_L', 'HANDLE_TYPE', 'KNOB_TYPE', 'PANEL_COLOR', 'CLEANING', 'HANDLE_COLOR', 'KNOB_COLOR', 'KNOB_STRUCTURE', 'NUMBER_OF_FUNCTIONS', 'OVEN_CONTROL', 'OVEN_TYPE', 'PIZZA_TRAY', 'TURNSPIT', 'ENERGY_CLASS', 'PANEL_TYPE', 'WIFI', 'BAKING_TRAY', 'PACKAGING_TYPE', 'POWER_CABLE', 'ROASTING_TRAY', 'TELESCOPIC_RAILS', 'WIRE_SHELF']

Ground Truth:
['REQUEST_ID', 'CHARACTERISTIC', 'VALUE', 'VALUE_DEFINITION', 'OPERATOR', 'REQUIREMENT_TYPE']

Customer Requests:
['REQUEST_ID', 'LLM_TEXT', 'INPUT_TOKENS', 'OUTPUT_TOKENS']

=== REQUEST ID VALIDATION ===
Unique Ground Truth IDs: 3000
Unique Customer IDs: 3000
Missing customer texts: 0
Customers without Ground Truth: 0

STEP 120: PASSED


### Step 121 — Data Quality and Requirement Coverage

**Objective:** Evaluate customer text quality, requirement distributions, and characteristic–value coverage.


In [9]:
# Step 121 - Data quality and requirement coverage analysis

print("=== CUSTOMER TEXT QUALITY ===")

texts = customer_requests["LLM_TEXT"].fillna("").astype(str).str.strip()

print("Total requests:", len(customer_requests))
print("Empty texts:", texts.eq("").sum())
print("Duplicate Request IDs:", customer_requests["REQUEST_ID"].duplicated().sum())
print("Duplicate texts:", texts.duplicated().sum())

print("\n=== REQUIREMENTS PER REQUEST ===")

req_counts = ground_truth.groupby("REQUEST_ID").size()

print(req_counts.describe())

print("\n=== REQUIREMENT TYPE DISTRIBUTION ===")

print(ground_truth["REQUIREMENT_TYPE"].value_counts())

print("\n=== OPERATOR DISTRIBUTION ===")

print(ground_truth["OPERATOR"].value_counts())

print("\n=== CHARACTERISTIC COVERAGE ===")

print("Unique characteristics:", ground_truth["CHARACTERISTIC"].nunique())

characteristic_coverage = (
    ground_truth.groupby("CHARACTERISTIC")
    .agg(
        REQUEST_COUNT=("REQUEST_ID", "nunique"),
        REQUIREMENT_COUNT=("REQUEST_ID", "size")
    )
    .sort_values("REQUEST_COUNT")
)

display(characteristic_coverage)

print("\n=== CHARACTERISTIC-VALUE COVERAGE ===")

value_coverage = (
    ground_truth.groupby(["CHARACTERISTIC", "VALUE"])
    .agg(
        REQUEST_COUNT=("REQUEST_ID", "nunique"),
        REQUIREMENT_COUNT=("REQUEST_ID", "size")
    )
    .sort_values("REQUEST_COUNT")
)

print("Unique characteristic-value pairs:", len(value_coverage))
print("Minimum request count:", value_coverage["REQUEST_COUNT"].min())
print("Pairs with fewer than 10 requests:",
      (value_coverage["REQUEST_COUNT"] < 10).sum())

display(value_coverage.head(15))

print("\nSTEP 121: COMPLETED")

=== CUSTOMER TEXT QUALITY ===
Total requests: 3000
Empty texts: 0
Duplicate Request IDs: 0
Duplicate texts: 0

=== REQUIREMENTS PER REQUEST ===
count    3000.000000
mean        4.007000
std         1.243436
min         2.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         6.000000
dtype: float64

=== REQUIREMENT TYPE DISTRIBUTION ===
REQUIREMENT_TYPE
MUST         8362
PREFERRED    3659
Name: count, dtype: int64

=== OPERATOR DISTRIBUTION ===
OPERATOR
EQUAL       7998
CONTAINS    4023
Name: count, dtype: int64

=== CHARACTERISTIC COVERAGE ===
Unique characteristics: 24


,REQUEST_COUNT,REQUIREMENT_COUNT
CHARACTERISTIC,,
POWER_CABLE,210,210
PACKAGING_TYPE,227,227
OVEN_VOLUME_L,268,268
KNOB_STRUCTURE,275,275
TURNSPIT,283,283
KNOB_COLOR,284,284
BAKING_TRAY,288,288
KNOB_TYPE,291,291
HANDLE_COLOR,299,299



=== CHARACTERISTIC-VALUE COVERAGE ===
Unique characteristic-value pairs: 99
Minimum request count: 32
Pairs with fewer than 10 requests: 0


REQUEST_COUNT  REQUIREMENT_COUNT
CHARACTERISTIC VALUE                                  
POWER_CABLE    V002              32                 32
               V005              37                 37
KNOB_TYPE      V004              40                 40
POWER_CABLE    V001              43                 43
OVEN_VOLUME_L  V001              44                 44
KNOB_COLOR     V004              45                 45
POWER_CABLE    V003              46                 46
HANDLE_COLOR   V004              50                 50
OVEN_VOLUME_L  V004              52                 52
POWER_CABLE    V004              52                 52
KNOB_TYPE      V003              55                 55
KNOB_COLOR     V002              56                 56
HANDLE_TYPE    V005              56                 56
KNOB_TYPE      V005              58                 58
ENERGY_CLASS   V003              60                 60


STEP 121: COMPLETED


### Step 122 — Requirement Operators and Oven Functions Validation

**Objective:** Examine requirement operators and validate the representation of oven functions in the Ground Truth dataset.

**Validation scope:**
- Analyze the distribution of EQUAL and CONTAINS operators.
- Identify which characteristics use each operator.
- Examine multiple oven function requirements within the same customer request.
- Check for duplicate or conflicting requirement records.

**Expected behavior:**
- EQUAL represents an exact characteristic–value requirement.
- CONTAINS represents a requirement that must be included among the product's supported functions.
- Multiple oven functions may be requested within a single customer request.

**Purpose:** Ensure that requirement operators are correctly represented before semantic validation and product matching.

In [10]:
# Step 122 - Validate requirement operators and oven functions

print("=== OPERATOR DISTRIBUTION ===")

operator_counts = (
    ground_truth["OPERATOR"]
    .fillna("MISSING")
    .astype(str)
    .str.strip()
    .value_counts()
)

print(operator_counts.to_string())

print("\n=== CHARACTERISTICS BY OPERATOR ===")

operator_matrix = pd.crosstab(
    ground_truth["CHARACTERISTIC"],
    ground_truth["OPERATOR"]
)

display(operator_matrix)

print("\n=== OVEN FUNCTIONS ANALYSIS ===")

oven_functions = ground_truth[
    ground_truth["CHARACTERISTIC"] == "OVEN_FUNCTIONS"
].copy()

print("Total oven function requirements:", len(oven_functions))
print("Requests containing oven functions:",
      oven_functions["REQUEST_ID"].nunique())

print("\nOven function operators:")
print(oven_functions["OPERATOR"].value_counts(dropna=False).to_string())

functions_per_request = (
    oven_functions.groupby("REQUEST_ID").size()
)

print("\nFunctions per request:")
print(functions_per_request.describe().to_string())

print("\nRequests with multiple oven functions:",
      (functions_per_request > 1).sum())

print("\n=== DUPLICATE REQUIREMENT CHECK ===")

duplicate_requirements = ground_truth.duplicated(
    subset=[
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "OPERATOR",
        "REQUIREMENT_TYPE"
    ]
).sum()

print("Exact duplicate requirements:", duplicate_requirements)

print("\n=== SAME CHARACTERISTIC-VALUE WITH DIFFERENT PRIORITIES ===")

priority_conflicts = (
    ground_truth.groupby(
        ["REQUEST_ID", "CHARACTERISTIC", "VALUE"]
    )["REQUIREMENT_TYPE"]
    .nunique()
)

print(
    "Characteristic-value pairs with conflicting priorities:",
    (priority_conflicts > 1).sum()
)

print("\n=== SAMPLE REQUESTS WITH MULTIPLE FUNCTIONS ===")

multi_function_ids = functions_per_request[
    functions_per_request > 1
].index[:5]

sample_functions = ground_truth[
    ground_truth["REQUEST_ID"].isin(multi_function_ids)
].sort_values(["REQUEST_ID", "CHARACTERISTIC", "VALUE"])

display(sample_functions)

print("\nSTEP 122: COMPLETED")

=== OPERATOR DISTRIBUTION ===
OPERATOR
EQUAL       7998
CONTAINS    4023

=== CHARACTERISTICS BY OPERATOR ===


OPERATOR,CONTAINS,EQUAL
CHARACTERISTIC,,
BAKING_TRAY,0,288
CLEANING,0,483
ENERGY_CLASS,0,321
HANDLE_COLOR,0,299
HANDLE_TYPE,0,339
KNOB_COLOR,0,284
KNOB_STRUCTURE,0,275
KNOB_TYPE,0,291
NUMBER_OF_FUNCTIONS,0,505



=== OVEN FUNCTIONS ANALYSIS ===
Total oven function requirements: 4023
Requests containing oven functions: 1990

Oven function operators:
OPERATOR
CONTAINS    4023

Functions per request:
count    1990.000000
mean        2.021608
std         0.814669
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         3.000000

Requests with multiple oven functions: 1351

=== DUPLICATE REQUIREMENT CHECK ===
Exact duplicate requirements: 0

=== SAME CHARACTERISTIC-VALUE WITH DIFFERENT PRIORITIES ===
Characteristic-value pairs with conflicting priorities: 0

=== SAMPLE REQUESTS WITH MULTIPLE FUNCTIONS ===


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
4034,R1012,KNOB_TYPE,V005,K5,EQUAL,PREFERRED
4033,R1012,OVEN_FUNCTIONS,Defrost,Defrost,CONTAINS,MUST
4031,R1012,OVEN_FUNCTIONS,DoubleGrill+Fan+Turnspit,DoubleGrill+Fan+Turnspit,CONTAINS,PREFERRED
4032,R1012,OVEN_FUNCTIONS,Fan+Ring,Fan+Ring,CONTAINS,PREFERRED
4039,R1013,BAKING_TRAY,V001,0,EQUAL,MUST
4040,R1013,KNOB_COLOR,V001,BLACK,EQUAL,MUST
4038,R1013,NUMBER_OF_FUNCTIONS,V004,10,EQUAL,MUST
4037,R1013,OVEN_FUNCTIONS,3D,3D,CONTAINS,MUST
4036,R1013,OVEN_FUNCTIONS,DoubleGrill+Fan,DoubleGrill+Fan,CONTAINS,MUST
4035,R1013,OVEN_FUNCTIONS,Pizza,Pizza,CONTAINS,PREFERRED



STEP 122: COMPLETED


### Step 123 — Customer Text Diversity Analysis

**Objective:** Evaluate the linguistic diversity of the 3,000 LLM-generated customer requests.

**Validation scope:**
- Analyze customer request lengths and word counts.
- Identify frequently repeated sentence openings.
- Measure the diversity of the first three and five words.
- Inspect a random sample of customer requests.

**Purpose:** Identify repetitive language patterns that may affect NLP model generalization.

**Note:** This analysis evaluates linguistic diversity, not semantic consistency. Semantic consistency will be validated separately by comparing extracted requirements with the original Ground Truth.

In [ ]:
# Step 123 - Customer text diversity analysis

import re   #re means regular expression, used for text processing and pattern matching

texts = customer_requests["LLM_TEXT"].fillna("").astype(str).str.strip()

print("=== TEXT LENGTH ANALYSIS ===")

text_lengths = texts.str.len()
word_counts = texts.str.split().str.len()

print("\nCharacter counts:")
print(text_lengths.describe().round(2).to_string())

print("\nWord counts:")
print(word_counts.describe().round(2).to_string())

print("\n=== SENTENCE OPENING ANALYSIS ===")

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())

normalized_texts = texts.apply(normalize_text)

def first_n_words(text, n):
    return " ".join(text.split()[:n])

first_3 = normalized_texts.apply(lambda x: first_n_words(x, 3))
first_5 = normalized_texts.apply(lambda x: first_n_words(x, 5))

print("\nUnique first-3-word patterns:", first_3.nunique())
print("Unique first-5-word patterns:", first_5.nunique())

print("\nTop 15 sentence openings (3 words):")
display(
    first_3.value_counts()
    .head(15)
    .rename_axis("OPENING")
    .reset_index(name="COUNT")
)

print("\nTop 15 sentence openings (5 words):")
display(
    first_5.value_counts()
    .head(15)
    .rename_axis("OPENING")
    .reset_index(name="COUNT")
)

print("\n=== NORMALIZED DUPLICATE TEXTS ===")

print("Duplicate normalized texts:", normalized_texts.duplicated().sum())

print("\n=== RANDOM CUSTOMER REQUEST SAMPLE ===")

sample_requests = customer_requests[
    ["REQUEST_ID", "LLM_TEXT"]
].sample(n=15, random_state=42)

for _, row in sample_requests.iterrows():
    print(f"\n{row['REQUEST_ID']}:")
    print(row["LLM_TEXT"])

print("\nSTEP 123: COMPLETED")

=== TEXT LENGTH ANALYSIS ===

Character counts:
count    3000.00
mean      154.86
std        40.85
min        47.00
25%       126.00
50%       153.00
75%       182.00
max       318.00

Word counts:
count    3000.00
mean       26.78
std         7.06
min        10.00
25%        22.00
50%        26.00
75%        31.00
max        56.00

=== SENTENCE OPENING ANALYSIS ===

Unique first-3-word patterns: 5
Unique first-5-word patterns: 14

Top 15 sentence openings (3 words):


,OPENING,COUNT
0,i m looking,2959
1,i need an,27
2,i need a,10
3,i want an,3
4,hi i need,1



Top 15 sentence openings (5 words):


,OPENING,COUNT
0,i m looking for an,1592
1,i m looking for a,1364
2,i need an oven that,21
3,i need an oven with,6
4,i need a convection oven,5
5,i m looking to buy,3
6,i need a 70 cm,2
7,i need a built in,1
8,i want an oven without,1
9,i want an oven with,1



=== NORMALIZED DUPLICATE TEXTS ===
Duplicate normalized texts: 0

=== RANDOM CUSTOMER REQUEST SAMPLE ===

R1802:
I'm looking for an oven that has a turbo function and both top and bottom heating elements, includes a pizza tray, and preferably comes without any wire shelves.

R1191:
I'm looking for an oven that definitely has a turbo function, and ideally it's a convection model—please show me options that meet that.

R1818:
I'm looking for a 60 cm built-in oven with a glass control panel that definitely has Turbo and a Double Grill with Fan functions; an energy class A model would be preferred.

R0245:
I'm looking for a convection oven with two telescopic rails, built-in Wi‑Fi, and a digital timer control.

R2506:
I'm looking for an oven that definitely has a double grill function, and preferably also offers a combined top, bottom and fan cooking mode; I would also like it to be energy class A.

R1118:
I'm looking for a built-in oven that has pyrolytic self-cleaning and includes a gri

### Step 124 — Semantic Validation Dataset Preparation

**Objective:** Prepare a representative sample of LLM-generated customer requests and their corresponding Ground Truth requirements for semantic consistency validation.

**Validation approach:**
- Select customer requests from the generated dataset.
- Retrieve the original technical requirements associated with each Request ID.
- Preserve characteristic codes, values, operators, and requirement priorities.
- Prepare structured records for independent requirement extraction.

**Semantic consistency criteria:**
1. All original MUST requirements must be preserved.
2. PREFERRED requirements must retain their priority.
3. Characteristic values must remain unchanged.
4. EQUAL and CONTAINS relationships must be preserved.
5. No additional technical requirements should be introduced.

**Methodology:** An LLM will independently extract structured requirements from the generated customer texts. These extracted requirements will be compared with the original Ground Truth.

**Important:** Structural validation does not guarantee semantic correctness. Automated semantic validation will be supplemented by manual review of selected examples.

**Expected output:** A validation sample containing customer texts and their original technical requirements.

In [12]:
# Step 124 - Prepare semantic validation sample

import json

# Select a reproducible sample of 30 customer requests
sample_size = 30

validation_sample = customer_requests[
    ["REQUEST_ID", "LLM_TEXT"]
].sample(
    n=sample_size,
    random_state=42
).copy()

# Include additional examples containing CONTAINS requirements
contains_ids = ground_truth.loc[
    ground_truth["OPERATOR"] == "CONTAINS",
    "REQUEST_ID"
].unique()

contains_sample = customer_requests[
    customer_requests["REQUEST_ID"].isin(contains_ids)
].sample(
    n=min(10, len(contains_ids)),
    random_state=123
)

# Combine samples and remove overlapping Request IDs
validation_sample = pd.concat(
    [validation_sample, contains_sample]
).drop_duplicates(
    subset="REQUEST_ID"
).reset_index(drop=True)

# Create structured validation records
validation_records = []

for _, row in validation_sample.iterrows():

    request_id = row["REQUEST_ID"]

    requirements = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ][
        [
            "CHARACTERISTIC",
            "VALUE",
            "VALUE_DEFINITION",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ].to_dict(orient="records")

    validation_records.append({
        "REQUEST_ID": request_id,
        "LLM_TEXT": row["LLM_TEXT"],
        "GROUND_TRUTH": requirements
    })

# Save validation input
validation_path = DATA_DIR / "interim" / "semantic_validation_sample.json"

validation_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(
        validation_records,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=== SEMANTIC VALIDATION SAMPLE ===")

print("Selected requests:", len(validation_records))

print(
    "Requests with CONTAINS:",
    len(
        set(validation_sample["REQUEST_ID"])
        & set(contains_ids)
    )
)

print("Saved:", validation_path)

print("\n=== FIRST VALIDATION EXAMPLE ===")

print("Request ID:", validation_records[0]["REQUEST_ID"])
print("Customer Text:", validation_records[0]["LLM_TEXT"])

print("\nGround Truth:")
display(
    pd.DataFrame(validation_records[0]["GROUND_TRUTH"])
)

print("\nSTEP 124: COMPLETED")

=== SEMANTIC VALIDATION SAMPLE ===
Selected requests: 40
Requests with CONTAINS: 30
Saved: /home/benal/oven-product-matching/data/interim/semantic_validation_sample.json

=== FIRST VALIDATION EXAMPLE ===
Request ID: R1802
Customer Text: I'm looking for an oven that has a turbo function and both top and bottom heating elements, includes a pizza tray, and preferably comes without any wire shelves.

Ground Truth:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,OVEN_FUNCTIONS,TURBO,TURBO,CONTAINS,MUST
1,OVEN_FUNCTIONS,Top+Bottom,Top+Bottom,CONTAINS,MUST
2,PIZZA_TRAY,V001,YES,EQUAL,MUST
3,WIRE_SHELF,V001,0,EQUAL,PREFERRED



STEP 124: COMPLETED


### Step 125 — Independent Semantic Extraction Pilot

**Objective:** Test whether an LLM can independently extract structured technical requirements from customer request texts.

**Methodology:**
- Select three customer requests from the semantic validation sample.
- Provide the LLM with customer texts and a technical vocabulary.
- Do not provide the original Ground Truth requirements for the selected requests.
- Extract characteristic, value, operator, and requirement priority.
- Compare the extracted requirements with Ground Truth in the next step.

**Validation criteria:**
- Correct characteristic identification.
- Correct value extraction.
- Correct EQUAL / CONTAINS operator assignment.
- Correct MUST / PREFERRED classification.
- No missing or additional requirements.

**Important:** The extraction model is used to assess dataset consistency. Its predictions are not automatically considered correct.

**Expected output:** Three structured extraction results for initial inspection.

In [14]:
# Step 125 - Independent semantic extraction pilot

import os       #os is a built-in Python module that provides a way to interact with the operating system, allowing you to access environment variables, file paths, and other OS-level functionality.
import json
from pathlib import Path

from dotenv import load_dotenv  #load_dotenv is a function from the python-dotenv library that loads environment variables from a .env file into the system's environment variables, allowing you to access sensitive information like API keys securely.
from openai import OpenAI

# Load API credentials
load_dotenv(PROJECT_DIR / ".env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Build technical vocabulary from the complete Ground Truth
vocabulary_df = (
    ground_truth[
        ["CHARACTERISTIC", "VALUE", "VALUE_DEFINITION"]
    ]
    .drop_duplicates()
    .sort_values(["CHARACTERISTIC", "VALUE"])
)

technical_vocabulary = vocabulary_df.to_dict(orient="records")

# Define extraction instructions
system_prompt = """
You extract structured technical requirements from English
customer requests about ovens.

Use only the provided technical vocabulary.

Return one record per explicitly requested characteristic-value pair.

For each requirement return:
- CHARACTERISTIC
- VALUE
- OPERATOR
- REQUIREMENT_TYPE

Rules:
1. MUST means mandatory.
2. PREFERRED means optional or desirable.
3. Use CONTAINS for OVEN_FUNCTIONS.
4. Use EQUAL for other characteristics.
5. Multiple oven functions must be separate records.
6. Do not invent requirements.
7. Do not infer requirements from general product knowledge.
8. Return valid JSON only, with this structure:

{
  "requirements": [
    {
      "CHARACTERISTIC": "...",
      "VALUE": "...",
      "OPERATOR": "...",
      "REQUIREMENT_TYPE": "..."
    }
  ]
}
"""

pilot_records = validation_records[:3]

extraction_results = []

for record in pilot_records:

    request_id = record["REQUEST_ID"]
    customer_text = record["LLM_TEXT"]

    user_prompt = json.dumps(
        {
            "technical_vocabulary": technical_vocabulary,
            "customer_text": customer_text
        },
        ensure_ascii=False
    )

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=system_prompt,
        input=user_prompt,
        reasoning={"effort": "minimal"},
        max_output_tokens=1500
    )

    if response.status != "completed":
        raise RuntimeError(
            f"{request_id}: API status = {response.status}"
        )

    raw_text = response.output_text.strip()

    if not raw_text:
        raise RuntimeError(
            f"{request_id}: Empty API response"
        )

    extracted = json.loads(raw_text)

    extraction_results.append(
        {
            "REQUEST_ID": request_id,
            "LLM_TEXT": customer_text,
            "EXTRACTED_REQUIREMENTS": extracted["requirements"],
            "INPUT_TOKENS": response.usage.input_tokens,
            "OUTPUT_TOKENS": response.usage.output_tokens
        }
    )

    print(f"\n=== {request_id} ===")
    print("Customer text:", customer_text)

    print("\nExtracted requirements:")
    display(pd.DataFrame(extracted["requirements"]))

    print("\nOriginal Ground Truth:")
    display(pd.DataFrame(record["GROUND_TRUTH"]))

print("\n=== PILOT SUMMARY ===")
print("Processed:", len(extraction_results))
print(
    "Input tokens:",
    sum(r["INPUT_TOKENS"] for r in extraction_results)
)
print(
    "Output tokens:",
    sum(r["OUTPUT_TOKENS"] for r in extraction_results)
)

print("\nSTEP 125: COMPLETED")


=== R1802 ===
Customer text: I'm looking for an oven that has a turbo function and both top and bottom heating elements, includes a pizza tray, and preferably comes without any wire shelves.

Extracted requirements:


,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,OVEN_FUNCTIONS,TURBO,CONTAINS,MUST
1,OVEN_FUNCTIONS,Top+Bottom,CONTAINS,MUST
2,PIZZA_TRAY,V001,EQUAL,MUST
3,WIRE_SHELF,V001,EQUAL,PREFERRED



Original Ground Truth:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,OVEN_FUNCTIONS,TURBO,TURBO,CONTAINS,MUST
1,OVEN_FUNCTIONS,Top+Bottom,Top+Bottom,CONTAINS,MUST
2,PIZZA_TRAY,V001,YES,EQUAL,MUST
3,WIRE_SHELF,V001,0,EQUAL,PREFERRED



=== R1191 ===
Customer text: I'm looking for an oven that definitely has a turbo function, and ideally it's a convection model—please show me options that meet that.

Extracted requirements:


,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,OVEN_FUNCTIONS,TURBO,CONTAINS,MUST
1,OVEN_TYPE,V001,EQUAL,PREFERRED


... (3 outputs left out)

,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,WIDTH_CM,V002,EQUAL,MUST
1,PANEL_TYPE,V001,EQUAL,MUST
2,OVEN_FUNCTIONS,TURBO,CONTAINS,MUST
3,OVEN_FUNCTIONS,DoubleGrill+Fan,CONTAINS,MUST
4,ENERGY_CLASS,V001,EQUAL,PREFERRED



Original Ground Truth:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,OVEN_FUNCTIONS,TURBO,TURBO,CONTAINS,MUST
1,OVEN_FUNCTIONS,DoubleGrill+Fan,DoubleGrill+Fan,CONTAINS,MUST
2,WIDTH_CM,V002,60,EQUAL,MUST
3,PANEL_TYPE,V001,GLASS,EQUAL,MUST
4,ENERGY_CLASS,V001,A,EQUAL,PREFERRED



=== PILOT SUMMARY ===
Processed: 3
Input tokens: 8719
Output tokens: 573

STEP 125: COMPLETED


### Step 126 — Automated Semantic Comparison

**Objective:** Compare independently extracted customer requirements with the original Ground Truth.

**Methodology:**
- Match requirements using CHARACTERISTIC, VALUE, OPERATOR, and REQUIREMENT_TYPE.
- Ignore row ordering.
- Identify missing, additional, or incorrectly classified requirements.
- Calculate exact-match accuracy and requirement-level Precision, Recall, and F1-score.

**Important:** These metrics measure agreement between LLM extraction and Ground Truth. They do not independently establish the semantic correctness of the original customer texts.

In [16]:
# Step 126 - Automated semantic comparison

import pandas as pd

COMPARE_COLUMNS = [
    "CHARACTERISTIC",
    "VALUE",
    "OPERATOR",
    "REQUIREMENT_TYPE"
]

def normalize_requirements(requirements):
    return {
        tuple(str(item[col]).strip() for col in COMPARE_COLUMNS)
        for item in requirements
    }


comparison_results = []

for result in extraction_results:

    request_id = result["REQUEST_ID"]

    predicted = normalize_requirements(
        result["EXTRACTED_REQUIREMENTS"]
    )

    expected_df = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ]

    expected = normalize_requirements(
        expected_df.to_dict(orient="records")
    )

    correct = predicted & expected
    missing = expected - predicted
    extra = predicted - expected

    comparison_results.append({
        "REQUEST_ID": request_id,
        "GROUND_TRUTH_COUNT": len(expected),
        "EXTRACTED_COUNT": len(predicted),
        "CORRECT": len(correct),
        "MISSING": len(missing),
        "EXTRA": len(extra),
        "EXACT_MATCH": predicted == expected,
        "MISSING_DETAILS": sorted(missing),
        "EXTRA_DETAILS": sorted(extra)
    })


comparison_df = pd.DataFrame(comparison_results)

tp = comparison_df["CORRECT"].sum()
fn = comparison_df["MISSING"].sum()
fp = comparison_df["EXTRA"].sum()

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("=== SEMANTIC COMPARISON ===")
print("Requests evaluated:", len(comparison_df))
print("Exact matches:", comparison_df["EXACT_MATCH"].sum())
print("Exact-match accuracy:", round(comparison_df["EXACT_MATCH"].mean(), 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

display(comparison_df)

print("\nSTEP 126: COMPLETED")

=== SEMANTIC COMPARISON ===
Requests evaluated: 3
Exact matches: 3
Exact-match accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0


,REQUEST_ID,GROUND_TRUTH_COUNT,EXTRACTED_COUNT,CORRECT,MISSING,EXTRA,EXACT_MATCH,MISSING_DETAILS,EXTRA_DETAILS
0,R1802,4,4,4,0,0,True,[],[]
1,R1191,2,2,2,0,0,True,[],[]
2,R1818,5,5,5,0,0,True,[],[]



STEP 126: COMPLETED


### Step 127 — Extended Semantic Validation

**Objective:** Extend independent semantic validation from 3 to 40 customer requests.

**Methodology:**
- Reuse the three completed pilot extractions.
- Extract structured requirements for the remaining requests.
- Use the technical vocabulary without revealing request-specific Ground Truth.
- Save each successful extraction to a checkpoint file.
- Preserve the original customer texts and request identifiers.

**Expected output:** Structured requirements for 40 customer requests, ready for comparison with Ground Truth.

**Note:** The extraction model is a validation instrument, not the NLP model being trained for the final recommendation system.

In [17]:
# Step 127 - Extended semantic validation with checkpointing

import json
import time
from pathlib import Path

checkpoint_path = Path(
    "../data/interim/semantic_extraction_40.json"
)

# Load previous results if a checkpoint exists
if checkpoint_path.exists():
    saved_results = json.loads(
        checkpoint_path.read_text(encoding="utf-8")
    )
else:
    saved_results = []

# Merge previously completed pilot results
results_by_id = {
    r["REQUEST_ID"]: r
    for r in saved_results
}

for r in extraction_results:
    results_by_id.setdefault(r["REQUEST_ID"], r)

# Process only requests not yet completed
for record in validation_records:

    request_id = record["REQUEST_ID"]

    if request_id in results_by_id:
        continue

    customer_text = record["LLM_TEXT"]

    user_prompt = json.dumps(
        {
            "technical_vocabulary": technical_vocabulary,
            "customer_text": customer_text
        },
        ensure_ascii=False
    )

    try:
        response = client.responses.create(
            model="gpt-5-mini",
            instructions=system_prompt,
            input=user_prompt,
            reasoning={"effort": "minimal"},
            max_output_tokens=1500
        )

        if response.status != "completed":
            print(f"Skipped {request_id}: {response.status}")
            continue

        raw_text = response.output_text.strip()

        if not raw_text:
            print(f"Skipped {request_id}: empty response")
            continue

        extracted = json.loads(raw_text)

        if (
            not isinstance(extracted, dict)
            or not isinstance(extracted.get("requirements"), list)
        ):
            print(f"Skipped {request_id}: invalid JSON structure")
            continue

        results_by_id[request_id] = {
            "REQUEST_ID": request_id,
            "LLM_TEXT": customer_text,
            "EXTRACTED_REQUIREMENTS": extracted["requirements"],
            "INPUT_TOKENS": response.usage.input_tokens,
            "OUTPUT_TOKENS": response.usage.output_tokens
        }

        # Save after each successful request
        checkpoint_path.write_text(
            json.dumps(
                list(results_by_id.values()),
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        print(
            f"Completed: {request_id} | "
            f"Total: {len(results_by_id)}/40"
        )

        time.sleep(0.2)

    except Exception as e:
        print(f"Error for {request_id}: {e}")
        continue


# Final results
extraction_results_40 = list(results_by_id.values())

print("\n=== STEP 127 SUMMARY ===")
print("Selected requests:", len(validation_records))
print("Completed extractions:", len(extraction_results_40))
print(
    "Remaining:",
    len(validation_records) - len(extraction_results_40)
)
print("Saved:", checkpoint_path)

print("\nSTEP 127: COMPLETED")

Completed: R0245 | Total: 4/40
Completed: R2506 | Total: 5/40
Completed: R1118 | Total: 6/40
Completed: R1412 | Total: 7/40
Completed: R2114 | Total: 8/40
Completed: R0402 | Total: 9/40
Completed: R2580 | Total: 10/40
Completed: R2255 | Total: 11/40
Completed: R0046 | Total: 12/40
Completed: R0638 | Total: 13/40
Completed: R1764 | Total: 14/40
Completed: R2777 | Total: 15/40
Completed: R0894 | Total: 16/40
Completed: R1074 | Total: 17/40
Completed: R0466 | Total: 18/40
Completed: R2739 | Total: 19/40
Completed: R2547 | Total: 20/40
Completed: R0038 | Total: 21/40
Completed: R0283 | Total: 22/40
Completed: R0906 | Total: 23/40
Completed: R2084 | Total: 24/40
Completed: R1361 | Total: 25/40
Completed: R2896 | Total: 26/40
Completed: R2604 | Total: 27/40
Completed: R1227 | Total: 28/40
Completed: R0290 | Total: 29/40
Completed: R1242 | Total: 30/40
Completed: R1789 | Total: 31/40
Completed: R2784 | Total: 32/40
Completed: R2935 | Total: 33/40
Completed: R2658 | Total: 34/40
Completed: R20

### Step 128 — Semantic Validation Metrics

**Objective:** Evaluate semantic consistency between LLM-generated customer texts and their original Ground Truth labels.

**Methodology:**
- Compare extracted requirements against Ground Truth for 40 sampled requests.
- Match CHARACTERISTIC, VALUE, OPERATOR, and REQUIREMENT_TYPE.
- Ignore requirement ordering while preserving multiple values for the same characteristic.
- Calculate exact-match accuracy, Precision, Recall, and F1-score.
- Identify missing and additional requirements for error analysis.

**Interpretation:**
- Exact Match measures whether all requirements of a customer request were recovered correctly.
- Precision measures the proportion of extracted requirements matching Ground Truth.
- Recall measures the proportion of Ground Truth requirements recovered.
- F1-score summarizes Precision and Recall.

**Limitation:** These metrics measure agreement with an independent LLM extractor. They are not yet the performance metrics of our final NLP model.

In [18]:
# Step 128 - Evaluate semantic consistency on 40 requests

import pandas as pd
from collections import Counter

COMPARE_COLUMNS = [
    "CHARACTERISTIC",
    "VALUE",
    "OPERATOR",
    "REQUIREMENT_TYPE"
]

def normalize_requirements(requirements):
    return Counter(
        tuple(str(item[col]).strip() for col in COMPARE_COLUMNS)
        for item in requirements
    )


comparison_results_40 = []

for result in extraction_results_40:

    request_id = result["REQUEST_ID"]

    predicted = normalize_requirements(
        result["EXTRACTED_REQUIREMENTS"]
    )

    expected_df = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ]

    expected = normalize_requirements(
        expected_df.to_dict(orient="records")
    )

    correct = predicted & expected
    missing = expected - predicted
    extra = predicted - expected

    comparison_results_40.append({
        "REQUEST_ID": request_id,
        "GROUND_TRUTH_COUNT": sum(expected.values()),
        "EXTRACTED_COUNT": sum(predicted.values()),
        "CORRECT": sum(correct.values()),
        "MISSING": sum(missing.values()),
        "EXTRA": sum(extra.values()),
        "EXACT_MATCH": predicted == expected,
        "MISSING_DETAILS": sorted(missing.elements()),
        "EXTRA_DETAILS": sorted(extra.elements())
    })


comparison_df_40 = pd.DataFrame(comparison_results_40)

tp = comparison_df_40["CORRECT"].sum()
fn = comparison_df_40["MISSING"].sum()
fp = comparison_df_40["EXTRA"].sum()

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

exact_match = comparison_df_40["EXACT_MATCH"].mean()

print("=== SEMANTIC VALIDATION RESULTS ===")
print("Requests evaluated:", len(comparison_df_40))
print("Total Ground Truth requirements:", tp + fn)
print("Correctly extracted requirements:", tp)
print("Missing requirements:", fn)
print("Additional requirements:", fp)

print("\n=== PERFORMANCE METRICS ===")
print(f"Exact-match accuracy: {exact_match:.2%}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

print("\n=== REQUEST-LEVEL RESULTS ===")

display(
    comparison_df_40[
        [
            "REQUEST_ID",
            "GROUND_TRUTH_COUNT",
            "EXTRACTED_COUNT",
            "CORRECT",
            "MISSING",
            "EXTRA",
            "EXACT_MATCH"
        ]
    ]
)

print("\n=== MISMATCH DETAILS ===")

mismatches = comparison_df_40[
    ~comparison_df_40["EXACT_MATCH"]
]

if mismatches.empty:
    print("No mismatches detected.")
else:
    display(
        mismatches[
            [
                "REQUEST_ID",
                "MISSING_DETAILS",
                "EXTRA_DETAILS"
            ]
        ]
    )

print("\nSTEP 128: COMPLETED")

=== SEMANTIC VALIDATION RESULTS ===
Requests evaluated: 40
Total Ground Truth requirements: 157
Correctly extracted requirements: 141
Missing requirements: 16
Additional requirements: 16

=== PERFORMANCE METRICS ===
Exact-match accuracy: 70.00%
Precision: 0.8981
Recall: 0.8981
F1-score: 0.8981

=== REQUEST-LEVEL RESULTS ===


,REQUEST_ID,GROUND_TRUTH_COUNT,EXTRACTED_COUNT,CORRECT,MISSING,EXTRA,EXACT_MATCH
0,R1802,4,4,4,0,0,True
1,R1191,2,2,2,0,0,True
2,R1818,5,5,5,0,0,True
3,R0245,4,4,4,0,0,True
4,R2506,3,3,3,0,0,True
5,R1118,3,3,2,1,1,False
6,R1412,4,4,3,1,1,False
7,R2114,3,3,2,1,1,False
8,R0402,4,4,4,0,0,True
9,R2580,5,5,5,0,0,True



=== MISMATCH DETAILS ===


,REQUEST_ID,MISSING_DETAILS,EXTRA_DETAILS
5,R1118,"[(OVEN_FUNCTIONS, Pyrolysis, CONTAINS, MUST)]","[(CLEANING, V003, EQUAL, MUST)]"
6,R1412,"[(OVEN_FUNCTIONS, Top+Bottom, CONTAINS, PREFER...","[(OVEN_FUNCTIONS, Top+Bottom, CONTAINS, MUST)]"
7,R2114,"[(OVEN_FUNCTIONS, DoubleGrill+Fan+Turnspit, CO...","[(OVEN_FUNCTIONS, DoubleGrill+Fan+Turnspit, CO..."
12,R0638,"[(TURNSPIT, V001, EQUAL, PREFERRED)]","[(TURNSPIT, V001, EQUAL, MUST)]"
13,R1764,"[(OVEN_FUNCTIONS, Pyrolysis, CONTAINS, MUST)]","[(CLEANING, V003, EQUAL, MUST)]"
14,R2777,"[(OVEN_FUNCTIONS, Fan+Ring, CONTAINS, PREFERRE...","[(CLEANING, V003, EQUAL, MUST), (OVEN_FUNCTION..."
22,R0906,"[(BAKING_TRAY, V002, EQUAL, PREFERRED)]","[(BAKING_TRAY, V002, EQUAL, MUST)]"
25,R2896,"[(OVEN_FUNCTIONS, DoubleGrill+Turnspit, CONTAI...","[(OVEN_FUNCTIONS, DoubleGrill+Turnspit, CONTAI..."
26,R2604,"[(TURNSPIT, V002, EQUAL, PREFERRED)]","[(TURNSPIT, V002, EQUAL, MUST)]"
27,R1227,"[(OVEN_FUNCTIONS, Grill, CONTAINS, PREFERRED)]","[(OVEN_FUNCTIONS, Grill, CONTAINS, MUST)]"



STEP 128: COMPLETED


### Step 129 — Semantic Mismatch Diagnosis

**Objective:** Identify and classify disagreements between Ground Truth and LLM-extracted requirements.

**Methodology:**
- Select customer requests that failed exact-match validation.
- Display the original customer text.
- Compare Ground Truth and extracted requirements.
- Examine missing and additional requirement pairs.
- Determine whether mismatches involve characteristic, value, operator, or requirement priority.

**Expected output:** A detailed error analysis supporting decisions about dataset quality and possible corrections.

**Important:** A mismatch does not automatically mean the generated customer text is incorrect. Manual semantic review is required to distinguish text-generation errors from extraction errors.

In [19]:
# Step 129 - Inspect semantic mismatches

import pandas as pd

failed_requests = comparison_df_40[
    ~comparison_df_40["EXACT_MATCH"]
].copy()

print("=== SEMANTIC MISMATCH DIAGNOSIS ===")
print("Failed requests:", len(failed_requests))

results_lookup = {
    r["REQUEST_ID"]: r
    for r in extraction_results_40
}

# Inspect the first five mismatched requests
for request_id in failed_requests["REQUEST_ID"].head(5):

    result = results_lookup[request_id]

    print("\n" + "=" * 70)
    print("REQUEST ID:", request_id)

    print("\nCUSTOMER TEXT:")
    print(result["LLM_TEXT"])

    print("\nGROUND TRUTH:")
    expected_df = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ][
        [
            "CHARACTERISTIC",
            "VALUE",
            "VALUE_DEFINITION",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ]

    display(expected_df)

    print("\nLLM EXTRACTION:")
    extracted_df = pd.DataFrame(
        result["EXTRACTED_REQUIREMENTS"]
    )

    display(extracted_df)

    mismatch_row = failed_requests[
        failed_requests["REQUEST_ID"] == request_id
    ].iloc[0]

    print("\nMISSING REQUIREMENTS:")
    print(mismatch_row["MISSING_DETAILS"])

    print("\nADDITIONAL REQUIREMENTS:")
    print(mismatch_row["EXTRA_DETAILS"])

print("\nSTEP 129: COMPLETED")

=== SEMANTIC MISMATCH DIAGNOSIS ===
Failed requests: 12

REQUEST ID: R1118

CUSTOMER TEXT:
I'm looking for a built-in oven that has pyrolytic self-cleaning and includes a grill with a turnspit/rotisserie function; I would prefer it to be energy class A.

GROUND TRUTH:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
4471,OVEN_FUNCTIONS,Pyrolysis,Pyrolysis,CONTAINS,MUST
4472,OVEN_FUNCTIONS,Grill+Turnspit,Grill+Turnspit,CONTAINS,MUST
4473,ENERGY_CLASS,V001,A,EQUAL,PREFERRED



LLM EXTRACTION:


,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,CLEANING,V003,EQUAL,MUST
1,OVEN_FUNCTIONS,Grill+Turnspit,CONTAINS,MUST
2,ENERGY_CLASS,V001,EQUAL,PREFERRED



MISSING REQUIREMENTS:
[('OVEN_FUNCTIONS', 'Pyrolysis', 'CONTAINS', 'MUST')]

ADDITIONAL REQUIREMENTS:
[('CLEANING', 'V003', 'EQUAL', 'MUST')]

REQUEST ID: R1412

CUSTOMER TEXT:
I'm looking for an oven that offers both top and bottom heating, has exactly one baking tray, does not include a pizza tray if possible, and must have an H4-style handle.

GROUND TRUTH:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
5651,OVEN_FUNCTIONS,Top+Bottom,Top+Bottom,CONTAINS,PREFERRED
5652,BAKING_TRAY,V002,1,EQUAL,MUST
5653,PIZZA_TRAY,V002,NO,EQUAL,PREFERRED
5654,HANDLE_TYPE,V004,H4,EQUAL,MUST


... (11 outputs left out)

,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
7137,OVEN_FUNCTIONS,Pyrolysis,Pyrolysis,CONTAINS,MUST
7138,OVEN_FUNCTIONS,Defrost,Defrost,CONTAINS,PREFERRED
7139,OVEN_FUNCTIONS,Fan+Ring,Fan+Ring,CONTAINS,MUST
7140,POWER_CABLE,V003,EU TYPE PLUG,EQUAL,MUST



LLM EXTRACTION:


,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,CLEANING,V003,EQUAL,MUST
1,OVEN_FUNCTIONS,Fan+Ring,CONTAINS,MUST
2,OVEN_FUNCTIONS,Defrost,CONTAINS,PREFERRED
3,POWER_CABLE,V003,EQUAL,MUST



MISSING REQUIREMENTS:
[('OVEN_FUNCTIONS', 'Pyrolysis', 'CONTAINS', 'MUST')]

ADDITIONAL REQUIREMENTS:
[('CLEANING', 'V003', 'EQUAL', 'MUST')]

STEP 129: COMPLETED


### Step 130 — Configuration Rule Integration Assessment

**Objective:** Examine how existing configuration rules can support semantic
interpretation and product matching.

The analysis distinguishes three layers:

1. Customer intent extracted from natural language.
2. Technical relationships defined by configuration rules.
3. Product matching based on mandatory and preferred requirements.

Configuration rules will not be used to silently modify the original
ground truth. Raw NLP performance and rule-assisted matching performance
will be evaluated separately.

In [20]:
# Step 130 - Inspect existing rule engine

from pathlib import Path

rule_engine_path = Path("../src/rule_engine.py")

assert rule_engine_path.exists(), "Rule engine file not found."

source = rule_engine_path.read_text(encoding="utf-8")

keywords = [
    "CONFIGURATION_RULES",
    "Auto Select",
    "OVEN_FUNCTIONS",
    "CLEANING",
    "TURNSPIT",
]

print("=== RULE ENGINE INSPECTION ===")
print("File:", rule_engine_path)

for keyword in keywords:
    print(f"{keyword}: {'FOUND' if keyword.lower() in source.lower() else 'NOT FOUND'}")

print("\n=== MATCHING SOURCE LINES ===")

for line_no, line in enumerate(source.splitlines(), start=1):
    if any(keyword.lower() in line.lower() for keyword in keywords):
        print(f"{line_no}: {line[:160]}")

print("\nSTEP 130: INSPECTION COMPLETED")

=== RULE ENGINE INSPECTION ===
File: ../src/rule_engine.py
CONFIGURATION_RULES: FOUND
Auto Select: FOUND
OVEN_FUNCTIONS: NOT FOUND
CLEANING: NOT FOUND
TURNSPIT: NOT FOUND

=== MATCHING SOURCE LINES ===
9:     def __init__(self, characteristic_values, configuration_rules):
23:         for rule_id, group in configuration_rules.groupby("RuleId"):
61:         """Evaluate Auto Select, Enable and Disable rules."""
82:                 if action_type == "Auto Select":
94:                 f"Conflicting Auto Select rules for {characteristic}: "
108:                     f"Unknown Auto Select value: {characteristic}={selected}"
113:                     f"Auto Select / Disable conflict: {characteristic}"
116:             # Auto Select has priority over Enable.
158:                     f"Auto Select / INC conflict: {characteristic}"

STEP 130: INSPECTION COMPLETED


### Step 131 — Semantic Error Classification

**Objective:** Identify and categorize semantic mismatches between the original Ground Truth and LLM-extracted customer requirements.

The initial validation of 40 customer requests achieved an exact-match accuracy of 70% and an F1-score of 0.8981.

However, not all mismatches represent the same type of error.

This step classifies mismatches into three categories:

1. **Priority mismatch:** The characteristic and value are correctly identified, but MUST/PREFERRED is incorrect.
2. **Potential technical equivalence:** Different characteristic-value representations may describe the same technical requirement.
3. **Other semantic mismatch:** Missing, additional, or incorrectly interpreted requirements requiring further investigation.

The original Ground Truth will remain unchanged during this diagnostic step.

**Expected output:** A structured mismatch report that supports targeted semantic validation and subsequent dataset correction.

In [21]:
# Step 131 - Classify semantic mismatches

import json
from pathlib import Path
import pandas as pd

# Load semantic extraction results
extraction_path = Path("../data/interim/semantic_extraction_40.json")

with open(extraction_path, "r", encoding="utf-8") as f:
    extraction_data = json.load(f)

print("Extraction data type:", type(extraction_data).__name__)

if isinstance(extraction_data, list):
    print("Number of records:", len(extraction_data))
    print("First record keys:", list(extraction_data[0].keys()))

elif isinstance(extraction_data, dict):
    print("Top-level keys:", list(extraction_data.keys())[:10])

print("\nGround Truth columns:")
print(ground_truth.columns.tolist())

print("\nSTEP 131A: STRUCTURE INSPECTION COMPLETED")

Extraction data type: list
Number of records: 40
First record keys: ['REQUEST_ID', 'LLM_TEXT', 'EXTRACTED_REQUIREMENTS', 'INPUT_TOKENS', 'OUTPUT_TOKENS']

Ground Truth columns:
['REQUEST_ID', 'CHARACTERISTIC', 'VALUE', 'VALUE_DEFINITION', 'OPERATOR', 'REQUIREMENT_TYPE']

STEP 131A: STRUCTURE INSPECTION COMPLETED


### Step 131B — Semantic Mismatch Classification

**Objective:** Classify mismatches between Ground Truth and LLM-extracted requirements.

Each requirement is compared using four fields:
- CHARACTERISTIC
- VALUE
- OPERATOR
- REQUIREMENT_TYPE

Mismatch categories:
1. **PRIORITY_MISMATCH:** Same characteristic, value and operator, but different MUST/PREFERRED labels.
2. **POTENTIAL_EQUIVALENCE:** Different representations that may describe the same technical requirement.
3. **OTHER_MISMATCH:** Remaining differences requiring investigation.

Technical equivalence is not assumed to be correct without validation against configuration rules.

**Output:** A structured diagnostic report for the 40-request semantic validation sample.

In [22]:
# Step 131B - Classify semantic mismatches

from collections import Counter

KEY_FIELDS = [
    "CHARACTERISTIC",
    "VALUE",
    "OPERATOR",
    "REQUIREMENT_TYPE"
]

def requirement_key(req):
    return tuple(str(req[field]).strip() for field in KEY_FIELDS)

def classify_mismatches(ground_truth_df, extraction_records):

    results = []

    for record in extraction_records:

        request_id = record["REQUEST_ID"]

        gt_rows = ground_truth_df[
            ground_truth_df["REQUEST_ID"] == request_id
        ]

        gt_requirements = gt_rows[KEY_FIELDS].to_dict("records")
        extracted_requirements = record["EXTRACTED_REQUIREMENTS"]

        gt_counter = Counter(
            requirement_key(r) for r in gt_requirements
        )

        extracted_counter = Counter(
            requirement_key(r) for r in extracted_requirements
        )

        missing = list((gt_counter - extracted_counter).elements())
        extra = list((extracted_counter - gt_counter).elements())

        # Work with copies so that each mismatch is paired only once
        remaining_missing = missing.copy()
        remaining_extra = extra.copy()

        # 1. Detect priority mismatches
        for m in missing:

            matching_extra = next(
                (
                    e for e in remaining_extra
                    if e[:3] == m[:3]
                    and e[3] != m[3]
                ),
                None
            )

            if matching_extra and m in remaining_missing:

                results.append({
                    "REQUEST_ID": request_id,
                    "CATEGORY": "PRIORITY_MISMATCH",
                    "GROUND_TRUTH": m,
                    "EXTRACTED": matching_extra
                })

                remaining_missing.remove(m)
                remaining_extra.remove(matching_extra)

        # 2. Identify potential technical equivalence
        for m in remaining_missing.copy():

            matching_extra = next(
                (
                    e for e in remaining_extra
                    if (
                        m[0] == "OVEN_FUNCTIONS"
                        and m[1].lower() == "pyrolysis"
                        and e[0] == "CLEANING"
                        and e[3] == m[3]
                    )
                ),
                None
            )

            if matching_extra:

                results.append({
                    "REQUEST_ID": request_id,
                    "CATEGORY": "POTENTIAL_EQUIVALENCE",
                    "GROUND_TRUTH": m,
                    "EXTRACTED": matching_extra
                })

                remaining_missing.remove(m)
                remaining_extra.remove(matching_extra)

        # 3. Remaining mismatches
        for m in remaining_missing:

            results.append({
                "REQUEST_ID": request_id,
                "CATEGORY": "OTHER_MISMATCH_MISSING",
                "GROUND_TRUTH": m,
                "EXTRACTED": None
            })

        for e in remaining_extra:

            results.append({
                "REQUEST_ID": request_id,
                "CATEGORY": "OTHER_MISMATCH_EXTRA",
                "GROUND_TRUTH": None,
                "EXTRACTED": e
            })

    return pd.DataFrame(
        results,
        columns=[
            "REQUEST_ID",
            "CATEGORY",
            "GROUND_TRUTH",
            "EXTRACTED"
        ]
    )


mismatch_df = classify_mismatches(
    ground_truth,
    extraction_data
)

print("=== SEMANTIC MISMATCH CLASSIFICATION ===")
print("Total mismatch records:", len(mismatch_df))
print("Affected requests:", mismatch_df["REQUEST_ID"].nunique())

print("\n=== CATEGORY DISTRIBUTION ===")
print(mismatch_df["CATEGORY"].value_counts())

print("\n=== MISMATCH DETAILS ===")
display(mismatch_df)

# Save diagnostic report
output_path = "../data/interim/semantic_mismatch_classification.csv"

mismatch_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", output_path)
print("STEP 131B: COMPLETED")

=== SEMANTIC MISMATCH CLASSIFICATION ===
Total mismatch records: 17
Affected requests: 12

=== CATEGORY DISTRIBUTION ===
CATEGORY
PRIORITY_MISMATCH         13
POTENTIAL_EQUIVALENCE      2
OTHER_MISMATCH_MISSING     1
OTHER_MISMATCH_EXTRA       1
Name: count, dtype: int64

=== MISMATCH DETAILS ===


,REQUEST_ID,CATEGORY,GROUND_TRUTH,EXTRACTED
0,R1118,POTENTIAL_EQUIVALENCE,"(OVEN_FUNCTIONS, Pyrolysis, CONTAINS, MUST)","(CLEANING, V003, EQUAL, MUST)"
1,R1412,PRIORITY_MISMATCH,"(OVEN_FUNCTIONS, Top+Bottom, CONTAINS, PREFERRED)","(OVEN_FUNCTIONS, Top+Bottom, CONTAINS, MUST)"
2,R2114,PRIORITY_MISMATCH,"(OVEN_FUNCTIONS, DoubleGrill+Fan+Turnspit, CON...","(OVEN_FUNCTIONS, DoubleGrill+Fan+Turnspit, CON..."
3,R0638,PRIORITY_MISMATCH,"(TURNSPIT, V001, EQUAL, PREFERRED)","(TURNSPIT, V001, EQUAL, MUST)"
4,R1764,POTENTIAL_EQUIVALENCE,"(OVEN_FUNCTIONS, Pyrolysis, CONTAINS, MUST)","(CLEANING, V003, EQUAL, MUST)"
5,R2777,PRIORITY_MISMATCH,"(OVEN_FUNCTIONS, Grill, CONTAINS, PREFERRED)","(OVEN_FUNCTIONS, Grill, CONTAINS, MUST)"
6,R2777,PRIORITY_MISMATCH,"(OVEN_FUNCTIONS, Fan+Ring, CONTAINS, PREFERRED)","(OVEN_FUNCTIONS, Fan+Ring, CONTAINS, MUST)"
7,R2777,OTHER_MISMATCH_MISSING,"(OVEN_FUNCTIONS, Pyrolysis, CONTAINS, PREFERRED)",None
8,R2777,OTHER_MISMATCH_EXTRA,None,"(CLEANING, V003, EQUAL, MUST)"
9,R0906,PRIORITY_MISMATCH,"(BAKING_TRAY, V002, EQUAL, PREFERRED)","(BAKING_TRAY, V002, EQUAL, MUST)"



Saved: ../data/interim/semantic_mismatch_classification.csv
STEP 131B: COMPLETED


### Step 132 — Priority Mismatch Investigation

**Objective:** Investigate whether priority mismatches originate from customer text generation or requirement extraction.

The previous analysis identified 13 priority mismatches across the semantic validation sample.

For each mismatch, the original customer text, Ground Truth priority, and extracted priority will be reviewed together.

This step distinguishes between:

1. **Generation error:** The customer text does not preserve the original MUST/PREFERRED requirement.
2. **Extraction error:** The customer text preserves the requirement, but the extraction model assigns an incorrect priority.
3. **Ambiguous wording:** The customer text does not clearly indicate the intended priority.

**Output:** A focused diagnostic table for manual review before large-scale semantic validation.

In [23]:
# Step 132 - Investigate priority mismatches

priority_df = mismatch_df[
    mismatch_df["CATEGORY"] == "PRIORITY_MISMATCH"
].copy()

text_lookup = {
    record["REQUEST_ID"]: record["LLM_TEXT"]
    for record in extraction_data
}

priority_df["CUSTOMER_TEXT"] = priority_df["REQUEST_ID"].map(
    text_lookup
)

priority_df["CHARACTERISTIC"] = priority_df[
    "GROUND_TRUTH"
].apply(lambda x: x[0])

priority_df["VALUE"] = priority_df[
    "GROUND_TRUTH"
].apply(lambda x: x[1])

priority_df["GT_PRIORITY"] = priority_df[
    "GROUND_TRUTH"
].apply(lambda x: x[3])

priority_df["EXTRACTED_PRIORITY"] = priority_df[
    "EXTRACTED"
].apply(lambda x: x[3])

priority_review = priority_df[
    [
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "GT_PRIORITY",
        "EXTRACTED_PRIORITY",
        "CUSTOMER_TEXT"
    ]
].copy()

pd.set_option("display.max_colwidth", 250)

print("=== PRIORITY MISMATCH INVESTIGATION ===")
print("Total mismatches:", len(priority_review))
print(
    "Affected requests:",
    priority_review["REQUEST_ID"].nunique()
)

display(priority_review)

print("\nSTEP 132: COMPLETED")

=== PRIORITY MISMATCH INVESTIGATION ===
Total mismatches: 13
Affected requests: 10


,REQUEST_ID,CHARACTERISTIC,VALUE,GT_PRIORITY,EXTRACTED_PRIORITY,CUSTOMER_TEXT
1,R1412,OVEN_FUNCTIONS,Top+Bottom,PREFERRED,MUST,"I'm looking for an oven that offers both top and bottom heating, has exactly one baking tray, does not include a pizza tray if possible, and must have an H4-style handle."
2,R2114,OVEN_FUNCTIONS,DoubleGrill+Fan+Turnspit,PREFERRED,MUST,"I'm looking for an oven that definitely includes both a Grill with a turnspit and also features a Double Grill with a fan and turnspit, and it should offer exactly 8 cooking functions in total."
3,R0638,TURNSPIT,V001,PREFERRED,MUST,"I'm looking for a 60 cm wide oven with a stainless steel handle, no wire shelf included, and preferably an electronic touch timer; it should also have a turnspit feature."
5,R2777,OVEN_FUNCTIONS,Grill,PREFERRED,MUST,"I'm looking for a built-in oven that offers pyrolysis self-cleaning, includes a grill function and a fan-with-ring cooking mode, has exactly one telescopic rail, and features retractable control knobs."
6,R2777,OVEN_FUNCTIONS,Fan+Ring,PREFERRED,MUST,"I'm looking for a built-in oven that offers pyrolysis self-cleaning, includes a grill function and a fan-with-ring cooking mode, has exactly one telescopic rail, and features retractable control knobs."
9,R0906,BAKING_TRAY,V002,PREFERRED,MUST,"I'm looking for a built-in oven that comes with a single baking tray, two telescopic rails, no wire shelves, and offers ten cooking functions, and it should have a digital timer control."
10,R2896,OVEN_FUNCTIONS,Top+Bottom,PREFERRED,MUST,"I'm looking for a convection oven that includes top and bottom heating, a grill with a turnspit, and a double grill with turnspit capability."
11,R2896,OVEN_FUNCTIONS,Grill+Turnspit,PREFERRED,MUST,"I'm looking for a convection oven that includes top and bottom heating, a grill with a turnspit, and a double grill with turnspit capability."
12,R2896,OVEN_FUNCTIONS,DoubleGrill+Turnspit,PREFERRED,MUST,"I'm looking for a convection oven that includes top and bottom heating, a grill with a turnspit, and a double grill with turnspit capability."
13,R2604,TURNSPIT,V002,PREFERRED,MUST,I'm looking for an oven with a silver handle that has both a double grill with fan function and hydrociclen cleaning; preferably it can also do pizza but it must not have a rotisserie.



STEP 132: COMPLETED


### Step 133 — Semantic Priority Validation

This step investigates MUST/PREFERRED mismatches identified during semantic validation.

The objective is to determine whether each mismatch originates from:
- Ambiguous wording in the generated customer request,
- An incorrect interpretation by the extraction LLM,
- Or a genuine inconsistency between the customer text and ground truth.

The original ground truth is preserved. No automatic correction is applied at this stage.

The findings will guide targeted data corrections before NLP model training.

In [26]:
# Step 133 - Review priority mismatches with complete customer texts

import pandas as pd

assert "priority_review" in globals(), "Run Step 132 first."

required_columns = [
    "REQUEST_ID",
    "CHARACTERISTIC",
    "VALUE",
    "GT_PRIORITY",
    "EXTRACTED_PRIORITY",
    "CUSTOMER_TEXT"
]

assert set(required_columns).issubset(priority_review.columns)

pd.set_option("display.max_colwidth", None)

print("=== PRIORITY MISMATCH REVIEW ===")
print("Total mismatches:", len(priority_review))
print("Affected requests:", priority_review["REQUEST_ID"].nunique())

for request_id, group in priority_review.groupby("REQUEST_ID", sort=False):

    print("\n" + "=" * 80)
    print("REQUEST ID:", request_id)

    print("\nCUSTOMER TEXT:")
    print(group["CUSTOMER_TEXT"].iloc[0])

    print("\nPRIORITY DIFFERENCES:")
    display(
        group[
            [
                "CHARACTERISTIC",
                "VALUE",
                "GT_PRIORITY",
                "EXTRACTED_PRIORITY"
            ]
        ].reset_index(drop=True)
    )

print("\nSTEP 133: COMPLETED")

=== PRIORITY MISMATCH REVIEW ===
Total mismatches: 13
Affected requests: 10

REQUEST ID: R1412

CUSTOMER TEXT:
I'm looking for an oven that offers both top and bottom heating, has exactly one baking tray, does not include a pizza tray if possible, and must have an H4-style handle.

PRIORITY DIFFERENCES:


,CHARACTERISTIC,VALUE,GT_PRIORITY,EXTRACTED_PRIORITY
0,OVEN_FUNCTIONS,Top+Bottom,PREFERRED,MUST



REQUEST ID: R2114

CUSTOMER TEXT:
I'm looking for an oven that definitely includes both a Grill with a turnspit and also features a Double Grill with a fan and turnspit, and it should offer exactly 8 cooking functions in total.

PRIORITY DIFFERENCES:


,CHARACTERISTIC,VALUE,GT_PRIORITY,EXTRACTED_PRIORITY
0,OVEN_FUNCTIONS,DoubleGrill+Fan+Turnspit,PREFERRED,MUST



REQUEST ID: R0638

CUSTOMER TEXT:
I'm looking for a 60 cm wide oven with a stainless steel handle, no wire shelf included, and preferably an electronic touch timer; it should also have a turnspit feature.

PRIORITY DIFFERENCES:


,CHARACTERISTIC,VALUE,GT_PRIORITY,EXTRACTED_PRIORITY
0,TURNSPIT,V001,PREFERRED,MUST


... (11 outputs left out)

,CHARACTERISTIC,VALUE,GT_PRIORITY,EXTRACTED_PRIORITY
0,OVEN_FUNCTIONS,Top+Bottom,PREFERRED,MUST



REQUEST ID: R2005

CUSTOMER TEXT:
I'm looking for an oven that includes a defrost function, comes with exactly one baking tray, and does not have Wi‑Fi.

PRIORITY DIFFERENCES:


,CHARACTERISTIC,VALUE,GT_PRIORITY,EXTRACTED_PRIORITY
0,OVEN_FUNCTIONS,Defrost,PREFERRED,MUST



STEP 133: COMPLETED


### Step 134 — Prepare Targeted Semantic Corrections

The semantic review identified cases where PREFERRED requirements were
expressed as mandatory or ambiguous statements.

This step prepares the affected requests for targeted correction.

Correction principles:
- Preserve the original ground truth.
- Preserve characteristic values and operators.
- Explicitly distinguish MUST and PREFERRED requirements.
- Avoid introducing or removing requirements.
- Keep original customer texts unchanged until corrections are validated.

The correction candidates are saved separately for traceability.

In [28]:
# Step 134 - Prepare targeted semantic correction candidates

from pathlib import Path
import pandas as pd

required_columns = [
    "REQUEST_ID",
    "CHARACTERISTIC",
    "VALUE",
    "GT_PRIORITY",
    "EXTRACTED_PRIORITY",
    "CUSTOMER_TEXT"
]

assert set(required_columns).issubset(priority_review.columns)

correction_candidates = (
    priority_review[required_columns]
    .copy()
    .sort_values(["REQUEST_ID", "CHARACTERISTIC", "VALUE"])
    .reset_index(drop=True)
)

correction_candidates["REVIEW_STATUS"] = "PENDING"
correction_candidates["CORRECTION_ACTION"] = ""
correction_candidates["REVIEW_NOTES"] = ""

output_path = Path("../data/interim/priority_correction_candidates.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

correction_candidates.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=== SEMANTIC CORRECTION PREPARATION ===")
print("Mismatch records:", len(correction_candidates))
print("Affected requests:", correction_candidates["REQUEST_ID"].nunique())
print("Saved:", output_path)

display(
    correction_candidates[
        [
            "REQUEST_ID",
            "CHARACTERISTIC",
            "VALUE",
            "GT_PRIORITY",
            "EXTRACTED_PRIORITY",
            "REVIEW_STATUS"
        ]
    ]
)

print("\nSTEP 134: COMPLETED")

=== SEMANTIC CORRECTION PREPARATION ===
Mismatch records: 13
Affected requests: 10
Saved: ../data/interim/priority_correction_candidates.csv


,REQUEST_ID,CHARACTERISTIC,VALUE,GT_PRIORITY,EXTRACTED_PRIORITY,REVIEW_STATUS
0,R0638,TURNSPIT,V001,PREFERRED,MUST,PENDING
1,R0906,BAKING_TRAY,V002,PREFERRED,MUST,PENDING
2,R1227,OVEN_FUNCTIONS,Grill,PREFERRED,MUST,PENDING
3,R1412,OVEN_FUNCTIONS,Top+Bottom,PREFERRED,MUST,PENDING
4,R2005,OVEN_FUNCTIONS,Defrost,PREFERRED,MUST,PENDING
5,R2114,OVEN_FUNCTIONS,DoubleGrill+Fan+Turnspit,PREFERRED,MUST,PENDING
6,R2604,TURNSPIT,V002,PREFERRED,MUST,PENDING
7,R2658,OVEN_FUNCTIONS,Top+Bottom,PREFERRED,MUST,PENDING
8,R2777,OVEN_FUNCTIONS,Fan+Ring,PREFERRED,MUST,PENDING
9,R2777,OVEN_FUNCTIONS,Grill,PREFERRED,MUST,PENDING



STEP 134: COMPLETED


### Step 135 — Prepare Ground-Truth-Guided Correction Inputs

This step prepares complete semantic correction inputs for requests
with identified MUST/PREFERRED mismatches.

Each input contains:
- The original customer request text.
- All ground truth requirements for that request.
- The specific priority mismatches identified during validation.

The correction process must preserve all characteristic values,
operators, and requirement priorities.

No API calls or dataset modifications are performed in this step.

In [29]:
# Step 135 - Prepare complete correction inputs

import json
from pathlib import Path

affected_ids = set(correction_candidates["REQUEST_ID"])

correction_inputs = []

for request_id in sorted(affected_ids):

    gt_rows = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ]

    customer_row = customer_requests[
        customer_requests["REQUEST_ID"] == request_id
    ]

    mismatch_rows = correction_candidates[
        correction_candidates["REQUEST_ID"] == request_id
    ]

    assert len(customer_row) == 1
    assert len(gt_rows) > 0

    correction_inputs.append({
        "REQUEST_ID": request_id,

        "ORIGINAL_TEXT": customer_row["LLM_TEXT"].iloc[0],

        "GROUND_TRUTH": gt_rows[
            [
                "CHARACTERISTIC",
                "VALUE",
                "VALUE_DEFINITION",
                "OPERATOR",
                "REQUIREMENT_TYPE"
            ]
        ].to_dict(orient="records"),

        "PRIORITY_MISMATCHES": mismatch_rows[
            [
                "CHARACTERISTIC",
                "VALUE",
                "GT_PRIORITY",
                "EXTRACTED_PRIORITY"
            ]
        ].to_dict(orient="records")
    })

output_path = Path(
    "../data/interim/semantic_correction_inputs.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        correction_inputs,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=== SEMANTIC CORRECTION INPUTS ===")
print("Requests prepared:", len(correction_inputs))
print("Saved:", output_path)

print("\nFirst request:", correction_inputs[0]["REQUEST_ID"])
print("Ground truth requirements:",
      len(correction_inputs[0]["GROUND_TRUTH"]))

print("\nSTEP 135: COMPLETED")

=== SEMANTIC CORRECTION INPUTS ===
Requests prepared: 10
Saved: ../data/interim/semantic_correction_inputs.json

First request: R0638
Ground truth requirements: 5

STEP 135: COMPLETED


### Step 136 — Ground-Truth-Guided Semantic Correction

This step uses an LLM to correct customer requests with identified
MUST/PREFERRED mismatches.

The correction is guided by the complete ground truth for each request.

Rules:
- Preserve all requirements and their values.
- Preserve MUST and PREFERRED priorities.
- Preserve CONTAINS and EQUAL semantics.
- Do not introduce additional requirements.
- Use natural English customer language.

Corrected texts are saved separately and require validation before
being incorporated into the final dataset.

In [30]:
# Step 136 - Correct priority mismatches using the OpenAI API

import os
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI

client = OpenAI()

# Use the same model as the previous generation step if possible.
MODEL = "gpt-4o-mini"

input_path = Path("../data/interim/semantic_correction_inputs.json")
output_path = Path("../data/interim/semantic_corrections_10.csv")

with open(input_path, "r", encoding="utf-8") as f:
    correction_inputs = json.load(f)

existing = {}

if output_path.exists():
    previous = pd.read_csv(output_path)

    existing = {
        row["REQUEST_ID"]: row
        for _, row in previous.iterrows()
    }

system_prompt = """
You are correcting synthetic customer requests for an oven recommendation dataset.

Your task is to rewrite the provided customer request so that it matches
the supplied ground truth exactly.

Rules:
1. Preserve every ground truth requirement.
2. Do not add any new product requirements.
3. MUST requirements must be clearly mandatory.
4. PREFERRED requirements must be clearly optional.
5. Preserve positive and negative meanings.
6. Preserve numerical values and quantities.
7. CONTAINS means that the oven must include the specified function,
   not that it must have only that function.
8. Use natural, concise English.
9. Return only the corrected customer request text.

For PREFERRED requirements, use clear expressions such as:
- "Ideally, ..."
- "I would prefer ..."
- "It would be nice if ..."
- "If possible, ..."

Do not turn preferences into mandatory conditions.
"""

results = list(existing.values())

for record in correction_inputs:

    request_id = record["REQUEST_ID"]

    if request_id in existing:
        print("Already corrected:", request_id)
        continue

    user_prompt = json.dumps(
        {
            "ORIGINAL_TEXT": record["ORIGINAL_TEXT"],
            "GROUND_TRUTH": record["GROUND_TRUTH"],
            "PRIORITY_MISMATCHES": record["PRIORITY_MISMATCHES"]
        },
        ensure_ascii=False,
        indent=2
    )

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.3,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    corrected_text = response.choices[0].message.content.strip()

    results.append(
        {
            "REQUEST_ID": request_id,
            "ORIGINAL_TEXT": record["ORIGINAL_TEXT"],
            "CORRECTED_TEXT": corrected_text,
            "INPUT_TOKENS": response.usage.prompt_tokens,
            "OUTPUT_TOKENS": response.usage.completion_tokens
        }
    )

    # Save after each request for safe resumption.
    pd.DataFrame(results).to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Corrected:", request_id)

corrected_df = pd.read_csv(output_path)

print("\n=== SEMANTIC CORRECTION SUMMARY ===")
print("Corrected requests:", len(corrected_df))
print("Empty texts:", corrected_df["CORRECTED_TEXT"].isna().sum())
print("Saved:", output_path)

display(
    corrected_df[
        ["REQUEST_ID", "ORIGINAL_TEXT", "CORRECTED_TEXT"]
    ].head(5)
)

print("\nSTEP 136: COMPLETED")

Corrected: R0638
Corrected: R0906
Corrected: R1227
Corrected: R1412
Corrected: R2005
Corrected: R2114
Corrected: R2604
Corrected: R2658
Corrected: R2777
Corrected: R2896

=== SEMANTIC CORRECTION SUMMARY ===
Corrected requests: 10
Empty texts: 0
Saved: ../data/interim/semantic_corrections_10.csv


,REQUEST_ID,ORIGINAL_TEXT,CORRECTED_TEXT
0,R0638,"I'm looking for a 60 cm wide oven with a stainless steel handle, no wire shelf included, and preferably an electronic touch timer; it should also have a turnspit feature.","I'm looking for a 60 cm wide oven with a stainless steel handle and no wire shelf included. Ideally, it would have a turnspit feature, and if possible, an electronic touch timer."
1,R0906,"I'm looking for a built-in oven that comes with a single baking tray, two telescopic rails, no wire shelves, and offers ten cooking functions, and it should have a digital timer control.","I'm looking for a built-in oven that must have two telescopic rails, no wire shelves, ten cooking functions, and a digital timer control. Ideally, it would come with a single baking tray."
2,R1227,"I'm looking for a stainless steel oven with an H5-style handle and a grill function, please.","I'm looking for a stainless steel oven that has an H5-style handle. Ideally, it should include a grill function, please."
3,R1412,"I'm looking for an oven that offers both top and bottom heating, has exactly one baking tray, does not include a pizza tray if possible, and must have an H4-style handle.","I'm looking for an oven that ideally offers both top and bottom heating, must have exactly one baking tray, and must have an H4-style handle. If possible, it would be nice if it does not include a pizza tray."
4,R2005,"I'm looking for an oven that includes a defrost function, comes with exactly one baking tray, and does not have Wi‑Fi.","I'm looking for an oven that ideally includes a defrost function, comes with exactly one baking tray, and does not have Wi-Fi."



STEP 136: COMPLETED


### Step 137 — Prepare Full Dataset Semantic Extraction

**Objective:** Extend the validated LLM-based requirement extraction process from 40 sampled customer requests to all 3,000 requests.

**Method:**
- Reuse the existing extraction prompt, technical vocabulary, and GPT-5-mini model.
- Extract structured requirements independently from each customer request.
- Preserve `REQUEST_ID` to enable comparison with the original ground truth.
- Save results incrementally using checkpointing.
- Resume interrupted processing without repeating completed API requests.

**Important:** This step performs requirement extraction only. Semantic accuracy will be evaluated separately by comparing the extracted requirements against the ground truth.

**Expected output:** 3,000 customer requests with structured LLM-extracted requirements.

In [31]:
# Step 137 - Prepare full semantic extraction

import json
from pathlib import Path

# Input and output paths
customer_requests_path = Path(
    "../data/interim/llm_customer_requests_3000.csv"
)

full_checkpoint_path = Path(
    "../data/interim/semantic_extraction_3000.json"
)

pilot_checkpoint_path = Path(
    "../data/interim/semantic_extraction_40.json"
)

# Use the customer request DataFrame loaded in Step 120
full_validation_records = (
    customer_requests[
        ["REQUEST_ID", "LLM_TEXT"]
    ]
    .to_dict(orient="records")
)

# Validate input
assert len(full_validation_records) == 3000

assert len({
    r["REQUEST_ID"]
    for r in full_validation_records
}) == 3000

# Load existing full checkpoint, if available
if full_checkpoint_path.exists():
    full_results = json.loads(
        full_checkpoint_path.read_text(encoding="utf-8")
    )
else:
    full_results = []

results_by_id_full = {
    r["REQUEST_ID"]: r
    for r in full_results
}

# Reuse previously extracted pilot results
if pilot_checkpoint_path.exists():

    pilot_results = json.loads(
        pilot_checkpoint_path.read_text(encoding="utf-8")
    )

    for record in pilot_results:
        results_by_id_full.setdefault(
            record["REQUEST_ID"],
            record
        )

# Identify remaining requests
all_request_ids = {
    r["REQUEST_ID"]
    for r in full_validation_records
}

completed_ids = set(results_by_id_full)

remaining_records = [
    r for r in full_validation_records
    if r["REQUEST_ID"] not in completed_ids
]

# Validate checkpoint IDs
assert completed_ids.issubset(all_request_ids)

# Save initial checkpoint
full_checkpoint_path.write_text(
    json.dumps(
        list(results_by_id_full.values()),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== FULL SEMANTIC EXTRACTION PREPARATION ===")
print("Total customer requests:", len(full_validation_records))
print("Already extracted:", len(results_by_id_full))
print("Remaining:", len(remaining_records))
print("Checkpoint:", full_checkpoint_path)

print("\nSTEP 137: PREPARATION COMPLETED")

=== FULL SEMANTIC EXTRACTION PREPARATION ===
Total customer requests: 3000
Already extracted: 40
Remaining: 2960
Checkpoint: ../data/interim/semantic_extraction_3000.json

STEP 137: PREPARATION COMPLETED


### Step 138 — Full Semantic Requirement Extraction

**Objective:** Extract structured technical requirements from all 3,000 customer requests using GPT-5-mini.

**Method:**
- Reuse the validated extraction prompt and technical vocabulary.
- Preserve previously completed extractions.
- Process remaining customer requests through the OpenAI API.
- Save results after every successful request.
- Support interruption and resumption through checkpointing.

**Output:** `semantic_extraction_3000.json`

**Next step:** Compare extracted requirements against ground truth and identify semantic inconsistencies.

In [37]:
# Step 138 - Full semantic extraction with checkpointing

import json
import time

# Number of requests to process in this execution
# Set to None to process all remaining requests
MAX_NEW_REQUESTS = None

# Reload checkpoint
saved_results = json.loads(
    full_checkpoint_path.read_text(encoding="utf-8")
)

results_by_id_full = {
    r["REQUEST_ID"]: r
    for r in saved_results
}

# Identify requests not yet processed
pending_records = [
    record
    for record in full_validation_records
    if record["REQUEST_ID"] not in results_by_id_full
]

if MAX_NEW_REQUESTS is not None:
    pending_records = pending_records[:MAX_NEW_REQUESTS]

print("=== FULL SEMANTIC EXTRACTION ===")
print("Total requests:", len(full_validation_records))
print("Already completed:", len(results_by_id_full))
print("Processing now:", len(pending_records))

# Process pending requests
for record in pending_records:

    request_id = record["REQUEST_ID"]
    customer_text = record["LLM_TEXT"]

    user_prompt = json.dumps(
        {
            "technical_vocabulary": technical_vocabulary,
            "customer_text": customer_text
        },
        ensure_ascii=False
    )

    try:
        response = client.responses.create(
            model="gpt-5-mini",
            instructions=extraction_system_prompt,
            input=user_prompt,
            reasoning={"effort": "minimal"},
            max_output_tokens=1500
        )

        if response.status != "completed":
            print(f"Skipped {request_id}: {response.status}")
            continue

        raw_text = response.output_text.strip()

        if not raw_text:
            print(f"Skipped {request_id}: empty response")
            continue

        extracted = json.loads(raw_text)

        if (
            not isinstance(extracted, dict)
            or not isinstance(extracted.get("requirements"), list)
        ):
            print(f"Skipped {request_id}: invalid JSON structure")
            continue

        results_by_id_full[request_id] = {
            "REQUEST_ID": request_id,
            "LLM_TEXT": customer_text,
            "EXTRACTED_REQUIREMENTS": extracted["requirements"],
            "INPUT_TOKENS": response.usage.input_tokens,
            "OUTPUT_TOKENS": response.usage.output_tokens
        }

        # Save checkpoint after every successful request
        full_checkpoint_path.write_text(
            json.dumps(
                list(results_by_id_full.values()),
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        print(
            f"Completed: {request_id} | "
            f"Total: {len(results_by_id_full)}/3000"
        )

        time.sleep(0.2)

    except Exception as e:
        print(f"Error for {request_id}: {e}")
        continue

# Summary
remaining_count = (
    len(full_validation_records) - len(results_by_id_full)
)

print("\n=== STEP 138 SUMMARY ===")
print("Completed:", len(results_by_id_full))
print("Remaining:", remaining_count)
print("Checkpoint:", full_checkpoint_path)

print("\nSTEP 138: COMPLETED")

=== FULL SEMANTIC EXTRACTION ===
Total requests: 3000
Already completed: 50
Processing now: 2950


Completed: R0004 | Total: 51/3000
Completed: R0005 | Total: 52/3000
Completed: R0006 | Total: 53/3000
Completed: R0007 | Total: 54/3000
Completed: R0008 | Total: 55/3000
Completed: R0009 | Total: 56/3000
Completed: R0010 | Total: 57/3000
Completed: R0011 | Total: 58/3000
Completed: R0012 | Total: 59/3000
Completed: R0013 | Total: 60/3000
Completed: R0014 | Total: 61/3000
Completed: R0015 | Total: 62/3000
Completed: R0016 | Total: 63/3000
Completed: R0017 | Total: 64/3000
Completed: R0018 | Total: 65/3000
Completed: R0019 | Total: 66/3000
Completed: R0020 | Total: 67/3000
Completed: R0021 | Total: 68/3000
Completed: R0022 | Total: 69/3000
Completed: R0023 | Total: 70/3000
Completed: R0024 | Total: 71/3000
Completed: R0025 | Total: 72/3000
Completed: R0026 | Total: 73/3000
Completed: R0027 | Total: 74/3000
Completed: R0028 | Total: 75/3000
Completed: R0029 | Total: 76/3000
Completed: R0030 | Total: 77/3000
Completed: R0031 | Total: 78/3000
Completed: R0032 | Total: 79/3000
Completed: R00

### Step 138C — Restore the Original Extraction Prompt

**Objective:** Restore the validated requirement extraction instructions from Step 125.

**Issue identified:** Step 136 overwrote the shared `system_prompt` variable with customer text correction instructions.

**Solution:** Use a dedicated `extraction_system_prompt` variable to prevent conflicts between text correction and requirement extraction.

**Method:** Preserve the original extraction rules, technical vocabulary, and API model without modification.

In [35]:
# Step 138C - Restore the original extraction prompt

extraction_system_prompt = """
You extract structured technical requirements from English
customer requests about ovens.

Use only the provided technical vocabulary.

Return one record per explicitly requested characteristic-value pair.

For each requirement return:
- CHARACTERISTIC
- VALUE
- OPERATOR
- REQUIREMENT_TYPE

Rules:
1. MUST means mandatory.
2. PREFERRED means optional or desirable.
3. Use CONTAINS for OVEN_FUNCTIONS.
4. Use EQUAL for other characteristics.
5. Multiple oven functions must be separate records.
6. Do not invent requirements.
7. Do not infer requirements from general product knowledge.
8. Return valid JSON only, with this structure:

{
  "requirements": [
    {
      "CHARACTERISTIC": "...",
      "VALUE": "...",
      "OPERATOR": "...",
      "REQUIREMENT_TYPE": "..."
    }
  ]
}
"""

assert '"requirements"' in extraction_system_prompt
assert "CONTAINS" in extraction_system_prompt
assert "PREFERRED" in extraction_system_prompt

print("=== EXTRACTION PROMPT RESTORED ===")
print("Original Step 125 instructions preserved.")
print("Dedicated variable: extraction_system_prompt")
print("\nSTEP 138C: COMPLETED")

=== EXTRACTION PROMPT RESTORED ===
Original Step 125 instructions preserved.
Dedicated variable: extraction_system_prompt

STEP 138C: COMPLETED


### Step 139 — Full Dataset Semantic Validation

**Objective:** Evaluate the semantic consistency of all 3,000 customer requests by comparing LLM-extracted requirements against the original ground truth.

**Method:**
- Match extracted requirements with ground truth using `REQUEST_ID`.
- Compare `CHARACTERISTIC`, `VALUE`, `OPERATOR`, and `REQUIREMENT_TYPE`.
- Identify missing, additional, and mismatched requirements.
- Calculate exact-match accuracy, precision, recall, and F1-score.
- Save request-level validation results for further analysis.

**Important:** This is a strict structural comparison. Technically equivalent characteristic-value representations will be investigated separately.

**Expected output:** Semantic validation metrics and a list of customer requests requiring further review.

In [38]:
# Step 139 - Full dataset semantic validation

import json
import pandas as pd
from pathlib import Path

extraction_path = Path(
    "../data/interim/semantic_extraction_3000.json"
)

output_path = Path(
    "../data/interim/semantic_validation_3000.csv"
)

# Load extraction results
with open(extraction_path, "r", encoding="utf-8") as f:
    extraction_data = json.load(f)

# Index extraction results by REQUEST_ID
extraction_by_id = {
    record["REQUEST_ID"]: record
    for record in extraction_data
}

# Validate dataset completeness
assert len(extraction_by_id) == 3000, (
    f"Expected 3000 extractions, found {len(extraction_by_id)}"
)

assert set(extraction_by_id) == set(
    ground_truth["REQUEST_ID"].unique()
), "Extraction and Ground Truth REQUEST_ID mismatch"

comparison_fields = [
    "CHARACTERISTIC",
    "VALUE",
    "OPERATOR",
    "REQUIREMENT_TYPE"
]

def normalize_requirement(record):
    return tuple(
        str(record.get(field, "")).strip()
        for field in comparison_fields
    )

validation_results = []

total_correct = 0
total_missing = 0
total_extra = 0

for request_id, gt_group in ground_truth.groupby("REQUEST_ID"):

    gt_requirements = [
        normalize_requirement(record)
        for record in gt_group.to_dict(orient="records")
    ]

    extracted_requirements = [
        normalize_requirement(record)
        for record in extraction_by_id[
            request_id
        ]["EXTRACTED_REQUIREMENTS"]
    ]

    # Compare requirements as multisets to preserve duplicates
    from collections import Counter

    gt_counter = Counter(gt_requirements)
    extracted_counter = Counter(extracted_requirements)

    correct_counter = gt_counter & extracted_counter
    missing_counter = gt_counter - extracted_counter
    extra_counter = extracted_counter - gt_counter

    correct = sum(correct_counter.values())
    missing = sum(missing_counter.values())
    extra = sum(extra_counter.values())

    exact_match = (
        missing == 0
        and extra == 0
    )

    total_correct += correct
    total_missing += missing
    total_extra += extra

    validation_results.append({
        "REQUEST_ID": request_id,
        "GROUND_TRUTH_COUNT": len(gt_requirements),
        "EXTRACTED_COUNT": len(extracted_requirements),
        "CORRECT": correct,
        "MISSING": missing,
        "EXTRA": extra,
        "EXACT_MATCH": exact_match,
        "MISSING_REQUIREMENTS": str(list(missing_counter.elements())),
        "EXTRA_REQUIREMENTS": str(list(extra_counter.elements()))
    })

validation_df = pd.DataFrame(validation_results)

# Calculate metrics
precision = (
    total_correct / (total_correct + total_extra)
    if total_correct + total_extra > 0 else 0
)

recall = (
    total_correct / (total_correct + total_missing)
    if total_correct + total_missing > 0 else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0 else 0
)

exact_match_accuracy = validation_df["EXACT_MATCH"].mean()

# Save results
validation_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=== FULL SEMANTIC VALIDATION RESULTS ===")
print("Requests evaluated:", len(validation_df))
print("Total ground truth requirements:",
      total_correct + total_missing)
print("Correctly extracted requirements:", total_correct)
print("Missing requirements:", total_missing)
print("Additional requirements:", total_extra)

print("\n=== PERFORMANCE METRICS ===")
print(f"Exact-match accuracy: {exact_match_accuracy:.2%}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

print("\n=== VALIDATION STATUS ===")
print("Exact matches:", validation_df["EXACT_MATCH"].sum())
print("Requests requiring review:",
      (~validation_df["EXACT_MATCH"]).sum())

print("\nSaved:", output_path)

display(
    validation_df[
        ~validation_df["EXACT_MATCH"]
    ].head(15)
)

print("\nSTEP 139: COMPLETED")

=== FULL SEMANTIC VALIDATION RESULTS ===
Requests evaluated: 3000
Total ground truth requirements: 12021
Correctly extracted requirements: 10626
Missing requirements: 1395
Additional requirements: 1487

=== PERFORMANCE METRICS ===
Exact-match accuracy: 65.57%
Precision: 0.8772
Recall: 0.8840
F1-score: 0.8806

=== VALIDATION STATUS ===
Exact matches: 1967
Requests requiring review: 1033

Saved: ../data/interim/semantic_validation_3000.csv


,REQUEST_ID,GROUND_TRUTH_COUNT,EXTRACTED_COUNT,CORRECT,MISSING,EXTRA,EXACT_MATCH,MISSING_REQUIREMENTS,EXTRA_REQUIREMENTS
5,R0006,4,4,2,2,2,False,"[('TURNSPIT', 'V001', 'EQUAL', 'PREFERRED'), ('PANEL_COLOR', 'V004', 'EQUAL', 'MUST')]","[('PANEL_COLOR', 'V004', 'EQUAL', 'PREFERRED'), ('TURNSPIT', 'V001', 'EQUAL', 'MUST')]"
9,R0010,4,5,4,0,1,False,[],"[('OVEN_FUNCTIONS', 'Steam', 'CONTAINS', 'MUST')]"
10,R0011,5,5,4,1,1,False,"[('KNOB_TYPE', 'V004', 'EQUAL', 'MUST')]","[('KNOB_TYPE', 'V003', 'EQUAL', 'MUST')]"
12,R0013,3,3,2,1,1,False,"[('PANEL_COLOR', 'V001', 'EQUAL', 'PREFERRED')]","[('PANEL_COLOR', 'V001', 'EQUAL', 'MUST')]"
16,R0017,6,6,5,1,1,False,"[('ROASTING_TRAY', 'V003', 'EQUAL', 'PREFERRED')]","[('ROASTING_TRAY', 'V003', 'EQUAL', 'MUST')]"
19,R0020,5,5,3,2,2,False,"[('OVEN_TYPE', 'V003', 'EQUAL', 'PREFERRED'), ('KNOB_COLOR', 'V002', 'EQUAL', 'MUST')]","[('OVEN_TYPE', 'V003', 'EQUAL', 'MUST'), ('KNOB_COLOR', 'V002', 'EQUAL', 'PREFERRED')]"
21,R0022,4,4,3,1,1,False,"[('PANEL_TYPE', 'V001', 'EQUAL', 'PREFERRED')]","[('PANEL_TYPE', 'V001', 'EQUAL', 'MUST')]"
32,R0033,4,4,3,1,1,False,"[('PACKAGING_TYPE', 'V002', 'EQUAL', 'MUST')]","[('PACKAGING_TYPE', 'V002', 'EQUAL', 'PREFERRED')]"
33,R0034,2,3,1,1,2,False,"[('CLEANING', 'V001', 'EQUAL', 'MUST')]","[('CLEANING', 'V002', 'EQUAL', 'PREFERRED'), ('CLEANING', 'V004', 'EQUAL', 'PREFERRED')]"
41,R0042,2,2,1,1,1,False,"[('CLEANING', 'V003', 'EQUAL', 'MUST')]","[('OVEN_FUNCTIONS', 'Pyrolysis', 'CONTAINS', 'MUST')]"



STEP 139: COMPLETED


## Step 140 — Validated Dataset Coverage Analysis

**Objective:** Evaluate whether the semantically validated customer requests
provide sufficient characteristic-value coverage for NLP model training.

**Method:**
1. Select customer requests with exact semantic matches.
2. Compare characteristic-value coverage between the complete dataset
   and the validated subset.
3. Identify rare or missing characteristic-value combinations.
4. Examine the distribution of MUST and PREFERRED requirements.

**Expected Outcome:** Determine whether the validated subset is suitable
for initial NLP training without losing important technical requirements.

**Note:** Exact semantic agreement is treated as a provisional validation
criterion, not definitive proof of label correctness. The remaining requests
will be retained for subsequent error analysis and correction.

In [39]:
# Step 140 - Analyze validated dataset coverage

import pandas as pd
from pathlib import Path

# Load semantic validation results
validation_path = Path(
    "../data/interim/semantic_validation_3000.csv"
)

validation_df = pd.read_csv(validation_path)

print("=== VALIDATION DATA STRUCTURE ===")
print("Columns:", validation_df.columns.tolist())
print("Total records:", len(validation_df))

# Identify requests with exact semantic matches
validated_ids = set(
    validation_df.loc[
        validation_df["EXACT_MATCH"].astype(str).str.lower() == "true",
        "REQUEST_ID"
    ]
)

# Filter customer requests and ground truth
validated_requests = customer_requests[
    customer_requests["REQUEST_ID"].isin(validated_ids)
].copy()

validated_ground_truth = ground_truth[
    ground_truth["REQUEST_ID"].isin(validated_ids)
].copy()

print("\n=== VALIDATED DATASET ===")
print("Validated requests:", len(validated_requests))
print("Validated requirements:", len(validated_ground_truth))

# Characteristic-value coverage in the complete dataset
full_coverage = (
    ground_truth.groupby(["CHARACTERISTIC", "VALUE"])
    ["REQUEST_ID"]
    .nunique()
    .reset_index(name="FULL_REQUEST_COUNT")
)

# Characteristic-value coverage in validated requests
validated_coverage = (
    validated_ground_truth.groupby(["CHARACTERISTIC", "VALUE"])
    ["REQUEST_ID"]
    .nunique()
    .reset_index(name="VALIDATED_REQUEST_COUNT")
)

# Compare coverage
coverage_comparison = full_coverage.merge(
    validated_coverage,
    on=["CHARACTERISTIC", "VALUE"],
    how="left"
)

coverage_comparison["VALIDATED_REQUEST_COUNT"] = (
    coverage_comparison["VALIDATED_REQUEST_COUNT"]
    .fillna(0)
    .astype(int)
)

coverage_comparison["COVERAGE_RATIO"] = (
    coverage_comparison["VALIDATED_REQUEST_COUNT"]
    / coverage_comparison["FULL_REQUEST_COUNT"]
)

print("\n=== CHARACTERISTIC-VALUE COVERAGE ===")
print("Total characteristic-value pairs:", len(coverage_comparison))

print(
    "Pairs retained:",
    (coverage_comparison["VALIDATED_REQUEST_COUNT"] > 0).sum()
)

print(
    "Pairs completely missing:",
    (coverage_comparison["VALIDATED_REQUEST_COUNT"] == 0).sum()
)

print(
    "Pairs with fewer than 10 validated requests:",
    (coverage_comparison["VALIDATED_REQUEST_COUNT"] < 10).sum()
)

print("\n=== LOWEST COVERAGE VALUES ===")

display(
    coverage_comparison.sort_values(
        "VALIDATED_REQUEST_COUNT"
    ).head(20)
)

print("\n=== REQUIREMENT TYPE DISTRIBUTION ===")

display(
    validated_ground_truth[
        "REQUIREMENT_TYPE"
    ].value_counts()
)

print("\n=== OPERATOR DISTRIBUTION ===")

display(
    validated_ground_truth[
        "OPERATOR"
    ].value_counts()
)

print("\nSTEP 140: COMPLETED")

=== VALIDATION DATA STRUCTURE ===
Columns: ['REQUEST_ID', 'GROUND_TRUTH_COUNT', 'EXTRACTED_COUNT', 'CORRECT', 'MISSING', 'EXTRA', 'EXACT_MATCH', 'MISSING_REQUIREMENTS', 'EXTRA_REQUIREMENTS']
Total records: 3000

=== VALIDATED DATASET ===
Validated requests: 1967
Validated requirements: 7727

=== CHARACTERISTIC-VALUE COVERAGE ===
Total characteristic-value pairs: 99
Pairs retained: 99
Pairs completely missing: 0
Pairs with fewer than 10 validated requests: 0

=== LOWEST COVERAGE VALUES ===


,CHARACTERISTIC,VALUE,FULL_REQUEST_COUNT,VALIDATED_REQUEST_COUNT,COVERAGE_RATIO
78,POWER_CABLE,V002,32,13,0.406250
53,OVEN_FUNCTIONS,Pyrolysis,162,20,0.123457
81,POWER_CABLE,V005,37,21,0.567568
28,KNOB_TYPE,V004,40,23,0.575000
77,POWER_CABLE,V001,43,23,0.534884
22,KNOB_COLOR,V004,45,29,0.644444
25,KNOB_TYPE,V001,63,30,0.476190
80,POWER_CABLE,V004,52,30,0.576923
79,POWER_CABLE,V003,46,30,0.652174
65,OVEN_VOLUME_L,V004,52,32,0.615385



=== REQUIREMENT TYPE DISTRIBUTION ===


REQUIREMENT_TYPE
MUST         5834
PREFERRED    1893
Name: count, dtype: int64


=== OPERATOR DISTRIBUTION ===


OPERATOR
EQUAL       5201
CONTAINS    2526
Name: count, dtype: int64


STEP 140: COMPLETED


## Step 141 — Prepare NLP Modeling Dataset

**Objective:** Create a clean, reproducible dataset for NLP model development.

**Method:**
1. Select customer requests that passed exact semantic validation.
2. Preserve the original ground truth requirements.
3. Save validated customer texts and their corresponding requirements.
4. Retain non-matching requests separately for future review.
5. Verify request identifiers and dataset integrity.

**Expected Outcome:**
Two aligned datasets containing validated customer requests and
their structured technical requirements, ready for NLP modeling.

**Important:** Exact-match validation provides provisional confidence
in semantic consistency. It does not guarantee error-free annotations.

In [40]:
# Step 141 - Prepare NLP modeling datasets

from pathlib import Path
import pandas as pd

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Select validated customer requests
nlp_requests = customer_requests[
    customer_requests["REQUEST_ID"].isin(validated_ids)
].copy()

# Select corresponding ground truth requirements
nlp_ground_truth = ground_truth[
    ground_truth["REQUEST_ID"].isin(validated_ids)
].copy()

# Keep requests requiring further review
review_requests = customer_requests[
    ~customer_requests["REQUEST_ID"].isin(validated_ids)
].copy()

# Sort for reproducibility
nlp_requests = (
    nlp_requests
    .sort_values("REQUEST_ID")
    .reset_index(drop=True)
)

nlp_ground_truth = (
    nlp_ground_truth
    .sort_values(["REQUEST_ID", "CHARACTERISTIC", "VALUE"])
    .reset_index(drop=True)
)

review_requests = (
    review_requests
    .sort_values("REQUEST_ID")
    .reset_index(drop=True)
)

# Verify dataset integrity
assert nlp_requests["REQUEST_ID"].nunique() == 1967

assert (
    set(nlp_requests["REQUEST_ID"])
    == set(nlp_ground_truth["REQUEST_ID"])
)

assert nlp_requests["REQUEST_ID"].duplicated().sum() == 0

assert nlp_requests["LLM_TEXT"].notna().all()

assert (
    set(nlp_requests["REQUEST_ID"])
    .isdisjoint(set(review_requests["REQUEST_ID"]))
)

assert (
    len(nlp_requests) + len(review_requests)
    == len(customer_requests)
)

# Save datasets
requests_path = output_dir / "nlp_customer_requests.csv"
ground_truth_path = output_dir / "nlp_ground_truth.csv"
review_path = output_dir / "customer_requests_review.csv"

nlp_requests.to_csv(
    requests_path,
    index=False,
    encoding="utf-8-sig"
)

nlp_ground_truth.to_csv(
    ground_truth_path,
    index=False,
    encoding="utf-8-sig"
)

review_requests.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("=== NLP DATASET PREPARATION ===")

print("Validated customer requests:", len(nlp_requests))
print("Validated requirements:", len(nlp_ground_truth))
print("Requests awaiting review:", len(review_requests))

print("\n=== SAVED FILES ===")
print(requests_path)
print(ground_truth_path)
print(review_path)

print("\n=== SAMPLE CUSTOMER REQUESTS ===")

display(
    nlp_requests[
        ["REQUEST_ID", "LLM_TEXT"]
    ].head(5)
)

print("\nSTEP 141: COMPLETED")

=== NLP DATASET PREPARATION ===
Validated customer requests: 1967
Validated requirements: 7727
Requests awaiting review: 1033

=== SAVED FILES ===
../data/processed/nlp_customer_requests.csv
../data/processed/nlp_ground_truth.csv
../data/processed/customer_requests_review.csv

=== SAMPLE CUSTOMER REQUESTS ===


,REQUEST_ID,LLM_TEXT
0,R0001,I'm looking for a steam oven that's exactly 45 cm wide.
1,R0002,"I'm looking for an oven with a white control panel, packaged with shrink wrap, and with an internal capacity of 50 liters."
2,R0003,"I'm looking for a built-in combination oven about 60 cm wide with two baking trays and a glass front panel, and it must have the K2-style knobs; I’d prefer the control panel to be silver if possible."
3,R0004,"I'm looking for an oven that has the H2 style handle, offers exactly 8 cooking functions, and preferably comes with a black handle."
4,R0005,"I'm looking for a 90 cm oven with black control knobs, a single telescopic rail, and pyrolytic self-cleaning — ideally with rotary-style knobs if possible."



STEP 141: COMPLETED
